In [ ]:
TASK 1

In [ ]:
print("=" * 70)
print("STEP 0A: PACKAGES")
print("=" * 70)

print("\n[0A] Installing...")

%pip install -q joblib scikit-learn pyarrow

import gc
import json
import os
import sys
import urllib.request
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

import tensorflow as tf
from tensorflow import keras

print("\n[0A] Packages ready.")
print(f"[0A] TensorFlow version: {tf.__version__}")

print("\nSTEP 0A DONE")
print("=" * 70)

STEP 0A: PACKAGES

[0A] Installing...

[0A] Packages ready.
[0A] TensorFlow version: 2.20.0

STEP 0A DONE


In [ ]:
print("=" * 70)
print("STEP 0B: MOUNT DRIVE")
print("=" * 70)

print("\n[0B] Mounting Google Drive...")

from google.colab import drive

drive.mount("/content/drive")

print("[0B] Drive mounted.")

DRIVE_ROOT = Path("/content/drive/MyDrive/vayupath")
DATA_DIR = DRIVE_ROOT / "data"
MODEL_DIR = DRIVE_ROOT / "models"

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("\n[0B] Folders:")
print(f"     DATA_DIR  = {DATA_DIR}")
print(f"     MODEL_DIR = {MODEL_DIR}")

print("\nSTEP 0B DONE")
print("=" * 70)

STEP 0B: MOUNT DRIVE

[0B] Mounting Google Drive...
Mounted at /content/drive
[0B] Drive mounted.

[0B] Folders:
     DATA_DIR  = /content/drive/MyDrive/vayupath/data
     MODEL_DIR = /content/drive/MyDrive/vayupath/models

STEP 0B DONE


In [ ]:
print("=" * 70)
print("STEP 0C: GPU CHECK")
print("=" * 70)

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    for g in gpus:
        print(f"[0C] Found GPU: {g}")
    print("[0C] Good — LSTM training can use the free GPU.")
else:
    print("[0C] WARNING: No GPU found.")
    print("     Go to Runtime → Change runtime type → GPU, then re-run.")

print("\nSTEP 0C DONE")
print("=" * 70)

STEP 0C: GPU CHECK
[0C] WARNING: No GPU found.
     Go to Runtime → Change runtime type → GPU, then re-run.

STEP 0C DONE


In [ ]:
print("=" * 70)
print("STEP 1A: DOWNLOAD YEARLY CPCB FILES (DELHI ONLY)")
print("=" * 70)

import urllib.request

# Years for training
YEARS = [2019, 2020, 2021, 2022, 2023]

# Required pollutant columns
RAW_COLS = [
    "Station Name",
    "Timestamp",
    "PM2.5 (µg/m³)",
    "PM10 (µg/m³)",
    "NO2 (µg/m³)",
    "CO (mg/m³)",
    "Ozone (µg/m³)",
]

TMP_DIR = Path("/content/cpcb_tmp")
TMP_DIR.mkdir(parents=True, exist_ok=True)

delhi_year_files = []

for year in YEARS:

    cache = DATA_DIR / f"delhi_pollutants_{year}.parquet"
    delhi_year_files.append(cache)

    if cache.exists():
        print(
            f"\n[1A] {year}: already cached -> {cache.name} "
            f"({cache.stat().st_size / 1e6:.1f} MB)"
        )
        continue

    url = (
        f"https://github.com/Vonter/india-cpcb-aqi/releases/download/"
        f"{year}/cpcb-air-quality-{year}.parquet"
    )

    tmp = TMP_DIR / f"cpcb-air-quality-{year}.parquet"

    print(f"\n[1A] {year}: downloading...")
    print(f"     {url}")

    urllib.request.urlretrieve(url, tmp)

    print(
        f"     downloaded {tmp.stat().st_size / 1e6:.1f} MB"
    )

    print(f"[1A] {year}: reading and filtering to Delhi...")

    year_df = pd.read_parquet(
        tmp,
        columns=["City"] + RAW_COLS
    )

    delhi = year_df[
        year_df["City"]
        .astype(str)
        .str.strip()
        .str.lower()
        == "delhi"
    ].copy()

    delhi = delhi.drop(columns=["City"])

    delhi.to_parquet(cache, index=False)

    print(
        f"[1A] {year}: saved {len(delhi):,} Delhi rows -> "
        f"{cache.name} "
        f"({cache.stat().st_size / 1e6:.1f} MB)"
    )

    del year_df, delhi

    tmp.unlink(missing_ok=True)

    gc.collect()


print("\n[1A] Delhi cache files ready:")

for path in delhi_year_files:
    print(
        f"     {path.name:32s} "
        f"{path.stat().st_size / 1e6:6.1f} MB"
    )


print("\n[1A] First 5 records from each Delhi file:")

for path in delhi_year_files:

    sample = pd.read_parquet(path)

    print(
        f"\n----- {path.name} "
        f"({len(sample):,} rows) -----"
    )

    display(sample.head(5))

    del sample

gc.collect()

print("\nSTEP 1A DONE")
print("=" * 70)

STEP 1A: DOWNLOAD YEARLY CPCB FILES (DELHI ONLY)

[1A] 2019: downloading...
     https://github.com/Vonter/india-cpcb-aqi/releases/download/2019/cpcb-air-quality-2019.parquet
     downloaded 282.6 MB
[1A] 2019: reading and filtering to Delhi...
[1A] 2019: saved 1,363,514 Delhi rows -> delhi_pollutants_2019.parquet (12.6 MB)

[1A] 2020: downloading...
     https://github.com/Vonter/india-cpcb-aqi/releases/download/2020/cpcb-air-quality-2020.parquet
     downloaded 335.1 MB
[1A] 2020: reading and filtering to Delhi...
[1A] 2020: saved 1,367,306 Delhi rows -> delhi_pollutants_2020.parquet (12.4 MB)

[1A] 2021: downloading...
     https://github.com/Vonter/india-cpcb-aqi/releases/download/2021/cpcb-air-quality-2021.parquet
     downloaded 405.7 MB
[1A] 2021: reading and filtering to Delhi...
[1A] 2021: saved 1,363,334 Delhi rows -> delhi_pollutants_2021.parquet (12.7 MB)

[1A] 2022: downloading...
     https://github.com/Vonter/india-cpcb-aqi/releases/download/2022/cpcb-air-quality-2022.pa

,Station Name,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO2 (µg/m³),CO (mg/m³),Ozone (µg/m³)
0,"Alipur, Delhi - DPCC",2019-01-01 00:00:00+00:00,338.0,512.0,145.1,3.8,1.2
1,"Alipur, Delhi - DPCC",2019-01-01 00:15:00+00:00,338.0,512.0,165.1,4.8,1.8
2,"Alipur, Delhi - DPCC",2019-01-01 00:30:00+00:00,385.0,539.0,170.0,3.6,1.3
3,"Alipur, Delhi - DPCC",2019-01-01 00:45:00+00:00,385.0,539.0,147.4,3.3,NaN
4,"Alipur, Delhi - DPCC",2019-01-01 01:00:00+00:00,385.0,539.0,146.6,2.6,NaN



----- delhi_pollutants_2020.parquet (1,367,306 rows) -----


,Station Name,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO2 (µg/m³),CO (mg/m³),Ozone (µg/m³)
0,"Alipur, Delhi - DPCC",2020-01-01 00:00:00+00:00,279.0,344.0,62.1,2.0,1.8
1,"Alipur, Delhi - DPCC",2020-01-01 00:15:00+00:00,279.0,344.0,61.4,2.0,1.5
2,"Alipur, Delhi - DPCC",2020-01-01 00:30:00+00:00,261.0,312.0,NaN,2.0,2.0
3,"Alipur, Delhi - DPCC",2020-01-01 00:45:00+00:00,261.0,312.0,59.1,1.9,NaN
4,"Alipur, Delhi - DPCC",2020-01-01 01:00:00+00:00,261.0,312.0,55.4,1.9,NaN



----- delhi_pollutants_2021.parquet (1,363,334 rows) -----


...


----- delhi_pollutants_2022.parquet (1,364,216 rows) -----


,Station Name,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO2 (µg/m³),CO (mg/m³),Ozone (µg/m³)
0,"Alipur, Delhi - DPCC",2022-01-01 00:00:00+00:00,217.0,337.0,59.7,1.6,2.6
1,"Alipur, Delhi - DPCC",2022-01-01 00:15:00+00:00,217.0,337.0,56.3,1.6,2.4
2,"Alipur, Delhi - DPCC",2022-01-01 00:30:00+00:00,217.0,337.0,51.5,1.6,2.6
3,"Alipur, Delhi - DPCC",2022-01-01 00:45:00+00:00,227.0,360.0,48.3,1.7,NaN
4,"Alipur, Delhi - DPCC",2022-01-01 01:00:00+00:00,227.0,360.0,NaN,NaN,2.5



----- delhi_pollutants_2023.parquet (1,362,463 rows) -----


,Station Name,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO2 (µg/m³),CO (mg/m³),Ozone (µg/m³)
0,"Alipur, Delhi - DPCC",2023-01-01 00:00:00+00:00,134.0,NaN,31.9,0.9,3.3
1,"Alipur, Delhi - DPCC",2023-01-01 00:15:00+00:00,134.0,174.0,30.9,0.9,3.3
2,"Alipur, Delhi - DPCC",2023-01-01 00:30:00+00:00,134.0,174.0,29.7,0.9,4.8
3,"Alipur, Delhi - DPCC",2023-01-01 00:45:00+00:00,134.0,174.0,27.2,0.8,6.5
4,"Alipur, Delhi - DPCC",2023-01-01 01:00:00+00:00,112.0,138.0,25.7,0.8,NaN



STEP 1A DONE


In [ ]:
print("=" * 70)
print("STEP 1B: COMBINE DELHI YEARS")
print("=" * 70)

RENAME = {
    "Station Name": "station",
    "Timestamp": "timestamp",
    "PM2.5 (µg/m³)": "pm25",
    "PM10 (µg/m³)": "pm10",
    "NO2 (µg/m³)": "no2",
    "CO (mg/m³)": "co",
    "Ozone (µg/m³)": "o3",
}

POLLUTANT_COLS = ["pm25", "pm10", "no2", "co", "o3"]

parts = []
for path in delhi_year_files:
    part = pd.read_parquet(path).rename(columns=RENAME)
    print(f"[1B] {path.name}: {len(part):,} rows")
    parts.append(part)

df = pd.concat(parts, ignore_index=True)
del parts
gc.collect()

# Timestamps are UTC in the source. Convert to India time so
# "hour of day" matches real Delhi mornings and evenings.
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df["timestamp"] = df["timestamp"].dt.tz_convert("Asia/Kolkata").dt.tz_localize(None)

df = df.sort_values(["station", "timestamp"]).reset_index(drop=True)

print(f"\n[1B] Combined: {len(df):,} rows (still 15-minute resolution)")
print(f"     Stations : {df['station'].nunique()}")
print(f"     Range    : {df['timestamp'].min()}  ->  {df['timestamp'].max()}")
print("\n[1B] Sample:")
display(df.head(3))

print("\nSTEP 1B DONE")
print("=" * 70)

STEP 1B: COMBINE DELHI YEARS
[1B] delhi_pollutants_2019.parquet: 1,363,514 rows
[1B] delhi_pollutants_2020.parquet: 1,367,306 rows
[1B] delhi_pollutants_2021.parquet: 1,363,334 rows
[1B] delhi_pollutants_2022.parquet: 1,364,216 rows
[1B] delhi_pollutants_2023.parquet: 1,362,463 rows

[1B] Combined: 6,820,833 rows (still 15-minute resolution)
     Stations : 39
     Range    : 2019-01-01 05:30:00  ->  2024-01-01 05:15:00

[1B] Sample:


,station,timestamp,pm25,pm10,no2,co,o3
0,"Alipur, Delhi - DPCC",2019-01-01 05:30:00,338.0,512.0,145.1,3.8,1.2
1,"Alipur, Delhi - DPCC",2019-01-01 05:45:00,338.0,512.0,165.1,4.8,1.8
2,"Alipur, Delhi - DPCC",2019-01-01 06:00:00,385.0,539.0,170.0,3.6,1.3



STEP 1B DONE


In [ ]:
print("=" * 70)
print("STEP 1C: AGGREGATE TO HOURLY")
print("=" * 70)

before = len(df)

df["timestamp"] = df["timestamp"].dt.floor("h")

df = (
    df.groupby(
        ["station", "timestamp"],
        as_index=False
    )[POLLUTANT_COLS]
    .mean()
    .sort_values(["station", "timestamp"])
    .reset_index(drop=True)
)

print(
    f"\n[1C] {before:,} fifteen-minute rows  ->  "
    f"{len(df):,} hourly rows"
)

print(f"[1C] Stations: {df['station'].nunique()}")

print("\n[1C] Sample hourly rows:")

display(df.head(5))

print("\nSTEP 1C DONE")
print("=" * 70)

STEP 1C: AGGREGATE TO HOURLY

[1C] 6,820,833 fifteen-minute rows  ->  1,707,496 hourly rows
[1C] Stations: 39

[1C] Sample hourly rows:


,station,timestamp,pm25,pm10,no2,co,o3
0,"Alipur, Delhi - DPCC",2019-01-01 05:00:00,338.0,512.0,155.100,4.300,1.500
1,"Alipur, Delhi - DPCC",2019-01-01 06:00:00,385.0,539.0,152.775,2.975,1.300
2,"Alipur, Delhi - DPCC",2019-01-01 07:00:00,311.0,489.0,137.400,2.400,3.125
3,"Alipur, Delhi - DPCC",2019-01-01 08:00:00,285.0,444.0,119.725,1.750,9.400
4,"Alipur, Delhi - DPCC",2019-01-01 09:00:00,268.0,415.0,109.475,1.800,10.150



STEP 1C DONE


In [ ]:
print("=" * 70)
print("STEP 1D: INSPECT")
print("=" * 70)

print(f"\n[1D] Rows        : {len(df):,}")
print(f"     Stations    : {df['station'].nunique()}")
print(f"     Date range  : {df['timestamp'].min()}  ->  {df['timestamp'].max()}")

print("\n[1D] Missing % per pollutant:")
for col in POLLUTANT_COLS:
    pct = df[col].isna().mean() * 100
    print(f"     {col:6s}  {pct:5.1f}%")

print("\n[1D] Value ranges:")
print(f"     {'col':6s} {'min':>10} {'median':>10} {'max':>10}")
for col in POLLUTANT_COLS:
    s = df[col].dropna()
    print(f"     {col:6s} {s.min():10.1f} {s.median():10.1f} {s.max():10.1f}")

print("\n[1D] Rows per year:")
for year, count in df.groupby(df["timestamp"].dt.year).size().items():
    print(f"     {year}: {count:,}")

print("\n[1D] Stations:")
for name in sorted(df["station"].unique()):
    print(f"     - {name}")

print("\nSTEP 1D DONE")
print("=" * 70)

STEP 1D: INSPECT

[1D] Rows        : 1,707,496
     Stations    : 39
     Date range  : 2019-01-01 05:00:00  ->  2024-01-01 05:00:00

[1D] Missing % per pollutant:
     pm25      9.4%
     pm10     12.1%
     no2       9.8%
     co       10.2%
     o3       10.5%

[1D] Value ranges:
     col           min     median        max
     pm25          0.0       69.2     1000.0
     pm10          0.0      171.0     1000.0
     no2           0.0       28.8      499.8
     co            0.0        1.0       47.8
     o3            0.0       18.2      200.0

[1D] Rows per year:
     2019: 341,030
     2020: 342,389
     2021: 341,248
     2022: 341,492
     2023: 341,103
     2024: 234

[1D] Stations:
     - Alipur, Delhi - DPCC
     - Anand Vihar, Delhi - DPCC
     - Ashok Vihar, Delhi - DPCC
     - Aya Nagar, Delhi - IMD
     - Bawana, Delhi - DPCC
     - Burari Crossing, Delhi - IMD
     - CRRI Mathura Road, Delhi - IMD
     - Chandni Chowk, Delhi - IITM
     - DTU, Delhi - CPCB
     - Dr. Ka

In [ ]:
print("=" * 70)
print("STEP 2A: CLEAN INVALID VALUES")
print("=" * 70)

LIMITS = {
    "pm25": (0, 1000),
    "pm10": (0, 1500),
    "no2": (0, 700),
    "co": (0, 50),
    "o3": (0, 500),
}

print("\n[2A] Cleaning each pollutant...")

for col in POLLUTANT_COLS:

    before_na = int(df[col].isna().sum())

    # Known CPCB error sentinel
    df.loc[df[col] == 999, col] = np.nan

    low, high = LIMITS[col]

    bad = (
        df[col].notna()
        & ~df[col].between(low, high)
    )

    df.loc[bad, col] = np.nan

    after_na = int(df[col].isna().sum())

    print(
        f"     {col:6s}: removed "
        f"{after_na - before_na:,} bad values "
        f"(now {df[col].isna().mean() * 100:4.1f}% blank)"
    )


# Rows without PM2.5 cannot be used for the target later
before = len(df)

df = df[df["pm25"].notna()].copy()

print(
    f"\n[2A] Dropped rows with no PM2.5: "
    f"{before - len(df):,}"
)

print(f"[2A] Rows left: {len(df):,}")

print("\nSTEP 2A DONE")
print("=" * 70)

STEP 2A: CLEAN INVALID VALUES

[2A] Cleaning each pollutant...
     pm25  : removed 0 bad values (now  9.4% blank)
     pm10  : removed 2 bad values (now 12.1% blank)
     no2   : removed 0 bad values (now  9.8% blank)
     co    : removed 0 bad values (now 10.2% blank)
     o3    : removed 0 bad values (now 10.5% blank)

[2A] Dropped rows with no PM2.5: 160,226
[2A] Rows left: 1,547,270

STEP 2A DONE


In [ ]:
print("=" * 70)
print("STEP 2B: STATION COVERAGE")
print("=" * 70)

MIN_COVERAGE = 0.50

date_min = df["timestamp"].min()
date_max = df["timestamp"].max()

total_hours = len(
    pd.date_range(date_min, date_max, freq="h")
)

readings = df.groupby("station")["pm25"].count()
coverage = readings / total_hours

print(f"\n[2B] Date range: {date_min}  ->  {date_max}")
print(f"[2B] Hours in the date range: {total_hours:,}")

print("\n[2B] Coverage per station (share of hours with PM2.5):")

for name, pct in coverage.sort_values(ascending=False).items():
    flag = "keep" if pct >= MIN_COVERAGE else "DROP"

    print(
        f"     {name[:46]:46} "
        f"{pct * 100:5.1f}%   {flag}"
    )

good_stations = sorted(
    coverage[coverage >= MIN_COVERAGE].index
)

df = df[df["station"].isin(good_stations)].copy()

print(
    f"\n[2B] Keeping {len(good_stations)} "
    f"of {len(coverage)} stations."
)

print(f"[2B] Rows now: {len(df):,}")

if len(good_stations) < 5:
    raise RuntimeError(
        "Fewer than 5 stations survived. Check Step 2A."
    )

print("\nSTEP 2B DONE")
print("=" * 70)

STEP 2B: STATION COVERAGE

[2B] Date range: 2019-01-01 05:00:00  ->  2024-01-01 05:00:00
[2B] Hours in the date range: 43,825

[2B] Coverage per station (share of hours with PM2.5):
     Major Dhyan Chand National Stadium, Delhi - DP  98.0%   keep
     Dwarka-Sector 8, Delhi - DPCC                   98.0%   keep
     Rohini, Delhi - DPCC                            97.9%   keep
     Sri Aurobindo Marg, Delhi - DPCC                97.9%   keep
     Okhla Phase-2, Delhi - DPCC                     97.7%   keep
     NSIT Dwarka, Delhi - CPCB                       97.6%   keep
     Nehru Nagar, Delhi - DPCC                       97.4%   keep
     Bawana, Delhi - DPCC                            97.1%   keep
     Shadipur, Delhi - CPCB                          97.0%   keep
     CRRI Mathura Road, Delhi - IMD                  96.8%   keep
     Sonia Vihar, Delhi - DPCC                       96.5%   keep
     Patparganj, Delhi - DPCC                        96.5%   keep
     Jawaharlal Nehru Stad

In [ ]:
print("=" * 70)
print("STEP 2C: COMPLETE HOURLY TIMELINE")
print("=" * 70)

FFILL_LIMIT = 3

all_hours = pd.date_range(
    df["timestamp"].min(),
    df["timestamp"].max(),
    freq="h"
)

full_index = pd.MultiIndex.from_product(
    [good_stations, all_hours],
    names=["station", "timestamp"]
)

print(
    f"\n[2C] {len(good_stations)} stations x "
    f"{len(all_hours):,} hours = "
    f"{len(full_index):,} slots"
)

df = (
    df.set_index(["station", "timestamp"])
      .reindex(full_index)
      .reset_index()
)

print("\n[2C] Forward-filling gaps of up to 3 hours...")

for col in POLLUTANT_COLS:

    before = int(df[col].isna().sum())

    df[col] = (
        df.groupby("station")[col]
          .ffill(limit=FFILL_LIMIT)
    )

    after = int(df[col].isna().sum())

    print(
        f"     {col:6s}: filled {before - after:,} "
        f"| still blank {after:,}"
    )

print(
    f"\n[2C] Records now: {len(df):,} rows "
    f"({df['station'].nunique()} stations x "
    f"{df['timestamp'].nunique():,} hours)"
)

gc.collect()

print("\nSTEP 2C DONE")
print("=" * 70)

STEP 2C: COMPLETE HOURLY TIMELINE

[2C] 36 stations x 43,825 hours = 1,577,700 slots

[2C] Forward-filling gaps of up to 3 hours...
     pm25  : filled 22,777 | still blank 58,571
     pm10  : filled 27,817 | still blank 122,675
     no2   : filled 32,314 | still blank 84,938
     co    : filled 42,364 | still blank 85,881
     o3    : filled 39,018 | still blank 92,089

[2C] Records now: 1,577,700 rows (36 stations x 43,825 hours)

STEP 2C DONE


In [ ]:
print("=" * 70)
print("STEP 2D: COMPUTE CPCB AQI")
print("=" * 70)


def sub_index_array(values, breakpoints):
    """
    Map pollutant readings onto the 0-500 AQI scale.
    """

    values = np.asarray(values, dtype=np.float64)

    out = np.full(
        values.shape,
        np.nan,
        dtype=np.float64
    )

    valid = np.isfinite(values) & (values >= 0)

    for c_lo, c_hi, a_lo, a_hi in breakpoints:

        mask = (
            valid
            & np.isnan(out)
            & (values >= c_lo)
            & (values <= c_hi)
        )

        out[mask] = (
            a_lo
            + (a_hi - a_lo)
            * (values[mask] - c_lo)
            / (c_hi - c_lo)
        )

    top = breakpoints[-1][1]

    above = valid & (values > top)

    out[above] = 500.0

    return out


# CPCB breakpoints

BP_PM25 = [
    (0, 30, 0, 50),
    (30, 60, 51, 100),
    (60, 90, 101, 200),
    (90, 120, 201, 300),
    (120, 250, 301, 400),
    (250, 380, 401, 500),
]

BP_PM10 = [
    (0, 50, 0, 50),
    (50, 100, 51, 100),
    (100, 250, 101, 200),
    (250, 350, 201, 300),
    (350, 430, 301, 400),
    (430, 500, 401, 500),
]

BP_NO2 = [
    (0, 40, 0, 50),
    (40, 80, 51, 100),
    (80, 180, 101, 200),
    (180, 280, 201, 300),
    (280, 400, 301, 400),
    (400, 1000, 401, 500),
]

BP_CO = [
    (0, 1.0, 0, 50),
    (1.0, 2.0, 51, 100),
    (2.0, 10.0, 101, 200),
    (10.0, 17.0, 201, 300),
    (17.0, 34.0, 301, 400),
    (34.0, 50.0, 401, 500),
]

BP_O3 = [
    (0, 50, 0, 50),
    (50, 100, 51, 100),
    (100, 168, 101, 200),
    (168, 208, 201, 300),
    (208, 748, 301, 400),
    (748, 1000, 401, 500),
]


print("\n[2D] Computing sub-indices...")

si_pm25 = sub_index_array(df["pm25"], BP_PM25)
si_pm10 = sub_index_array(df["pm10"], BP_PM10)
si_no2 = sub_index_array(df["no2"], BP_NO2)
si_co = sub_index_array(df["co"], BP_CO)
si_o3 = sub_index_array(df["o3"], BP_O3)


# Worst sub-index becomes the AQI

df["aqi"] = np.nanmax(
    np.column_stack([
        si_pm25,
        si_pm10,
        si_no2,
        si_co,
        si_o3
    ]),
    axis=1
)

df.loc[np.isnan(si_pm25), "aqi"] = np.nan

df["aqi"] = df["aqi"].clip(upper=500)


print(
    f"[2D] Rows with AQI : "
    f"{df['aqi'].notna().sum():,} / {len(df):,}"
)

print(f"     unique values : {df['aqi'].nunique():,}")

print(
    f"     mean / std    : "
    f"{df['aqi'].mean():.1f} / "
    f"{df['aqi'].std():.1f}"
)

print(
    f"     min / max     : "
    f"{df['aqi'].min():.1f} / "
    f"{df['aqi'].max():.1f}"
)


print("\n[2D] AQI mean by year:")

for year, g in df.dropna(subset=["aqi"]).groupby(
    df["timestamp"].dt.year
):
    print(
        f"     {year}: mean "
        f"{g['aqi'].mean():6.1f}   "
        f"std {g['aqi'].std():6.1f}"
    )


# Save cleaned hourly table

hourly_path = DATA_DIR / "delhi_cpcb_hourly.csv"

save_cols = [
    "station",
    "timestamp"
] + POLLUTANT_COLS + ["aqi"]

df.loc[
    df["aqi"].notna(),
    save_cols
].to_csv(
    hourly_path,
    index=False
)

print(
    f"\n[2D] Saved cleaned table -> "
    f"{hourly_path}"
)


print("\n[2D] Sample rows (first 5 with AQI):")

display(
    df.loc[
        df["aqi"].notna(),
        save_cols
    ].head(5)
)


print("\nSTEP 2D DONE")
print("=" * 70)

STEP 2D: COMPUTE CPCB AQI

[2D] Computing sub-indices...


/tmp/ipykernel_840/1920247395.py:105: RuntimeWarning: All-NaN slice encountered
  df["aqi"] = np.nanmax(


[2D] Rows with AQI : 1,519,129 / 1,577,700
     unique values : 182,391
     mean / std    : 214.6 / 133.1
     min / max     : 1.7 / 500.0

[2D] AQI mean by year:
     2019: mean  223.7   std  134.1
     2020: mean  194.7   std  132.6
     2021: mean  218.1   std  136.4
     2022: mean  220.2   std  128.2
     2023: mean  216.3   std  132.2
     2024: mean  374.7   std   51.3

[2D] Saved cleaned table -> /content/drive/MyDrive/vayupath/data/delhi_cpcb_hourly.csv

[2D] Sample rows (first 5 with AQI):


,station,timestamp,pm25,pm10,no2,co,o3,aqi
0,"Alipur, Delhi - DPCC",2019-01-01 05:00:00,338.0,512.0,155.100,4.300,1.500,500.000000
1,"Alipur, Delhi - DPCC",2019-01-01 06:00:00,385.0,539.0,152.775,2.975,1.300,500.000000
2,"Alipur, Delhi - DPCC",2019-01-01 07:00:00,311.0,489.0,137.400,2.400,3.125,484.442857
3,"Alipur, Delhi - DPCC",2019-01-01 08:00:00,285.0,444.0,119.725,1.750,9.400,427.653846
4,"Alipur, Delhi - DPCC",2019-01-01 09:00:00,268.0,415.0,109.475,1.800,10.150,414.707692



STEP 2D DONE


In [ ]:
print("=" * 70)
print("STEP 2E: SPATIAL VARIATION CHECK")
print("=" * 70)

per_hour = (
    df.dropna(subset=["aqi"])
      .groupby("timestamp")["aqi"]
)

spread = (per_hour.max() - per_hour.min()).mean()
station_std = per_hour.std().mean()

print("\n[2E] Averaged over every hour:")
print(
    f"     gap between dirtiest and cleanest station : "
    f"{spread:.0f} AQI points"
)
print(
    f"     spread across stations (std)              : "
    f"{station_std:.0f} AQI points"
)

example_hour = (
    df.dropna(subset=["aqi"])
      .groupby("timestamp")
      .size()
      .idxmax()
)

snapshot = (
    df[df["timestamp"] == example_hour]
      .dropna(subset=["aqi"])
      .sort_values("aqi", ascending=False)
)

print(
    f"\n[2E] One real hour ({example_hour}) — "
    f"dirtiest and cleanest:"
)

for _, row in pd.concat(
    [snapshot.head(4), snapshot.tail(4)]
).iterrows():
    print(
        f"     {row['station'][:46]:46} "
        f"{row['aqi']:5.0f}"
    )

print("\n[2E] Average AQI per station:")

for name, value in (
    df.groupby("station")["aqi"]
      .mean()
      .sort_values(ascending=False)
      .items()
):
    print(
        f"     {name[:46]:46} "
        f"{value:5.0f}"
    )

if station_std < 5:
    raise RuntimeError(
        "Stations barely differ from each other. "
        "Route optimization needs real spatial variation — "
        "check the data source."
    )

print(
    "\n     OK: stations genuinely differ. "
    "Route comparison is possible."
)

print("\nSTEP 2E DONE")
print("=" * 70)

STEP 2E: SPATIAL VARIATION CHECK

[2E] Averaged over every hour:
     gap between dirtiest and cleanest station : 229 AQI points
     spread across stations (std)              : 52 AQI points

[2E] One real hour (2019-01-01 05:00:00) — dirtiest and cleanest:
     Alipur, Delhi - DPCC                             500
     Anand Vihar, Delhi - DPCC                        500
     Ashok Vihar, Delhi - DPCC                        500
     CRRI Mathura Road, Delhi - IMD                   500
     Narela, Delhi - DPCC                             422
     Sri Aurobindo Marg, Delhi - DPCC                 379
     Aya Nagar, Delhi - IMD                           373
     Najafgarh, Delhi - DPCC                          348

[2E] Average AQI per station:
     Anand Vihar, Delhi - DPCC                        264
     Mundka, Delhi - DPCC                             255
     Bawana, Delhi - DPCC                             244
     Jahangirpuri, Delhi - DPCC                       244
     Wazirpur,

In [ ]:
print("=" * 70)
print("STEP 3A: TIME FEATURES")
print("=" * 70)


def as_circle(values, period):
    """
    Turn a repeating value into sin/cos coordinates on a circle.
    """
    angle = 2 * np.pi * values / period
    return np.sin(angle), np.cos(angle)


ts = df["timestamp"]

df["hour_sin"], df["hour_cos"] = as_circle(
    ts.dt.hour, 24
)

df["dow_sin"], df["dow_cos"] = as_circle(
    ts.dt.dayofweek, 7
)

df["month_sin"], df["month_cos"] = as_circle(
    ts.dt.month - 1, 12
)


TIME_FEATURES = [
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos"
]


print(
    f"\n[3A] Added {len(TIME_FEATURES)} time features: "
    f"{TIME_FEATURES}"
)


print(
    "\n[3A] Check that midnight and 11pm "
    "end up close together:"
)

for hour in (22, 23, 0, 1):
    s, c = as_circle(
        np.array([hour]),
        24
    )

    print(
        f"     {hour:02d}:00  "
        f"sin={s[0]:+.2f}  "
        f"cos={c[0]:+.2f}"
    )


print("\n[3A] Dataset with time features:")
print(df.head())

print(
    f"\n[3A] Shape: "
    f"{df.shape[0]:,} rows × {df.shape[1]} columns"
)


print("\nSTEP 3A DONE")
print("=" * 70)

STEP 3A: TIME FEATURES

[3A] Added 6 time features: ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']

[3A] Check that midnight and 11pm end up close together:
     22:00  sin=-0.50  cos=+0.87
     23:00  sin=-0.26  cos=+0.97
     00:00  sin=+0.00  cos=+1.00
     01:00  sin=+0.26  cos=+0.97

[3A] Dataset with time features:
                station           timestamp   pm25   pm10      no2     co  \
0  Alipur, Delhi - DPCC 2019-01-01 05:00:00  338.0  512.0  155.100  4.300   
1  Alipur, Delhi - DPCC 2019-01-01 06:00:00  385.0  539.0  152.775  2.975   
2  Alipur, Delhi - DPCC 2019-01-01 07:00:00  311.0  489.0  137.400  2.400   
3  Alipur, Delhi - DPCC 2019-01-01 08:00:00  285.0  444.0  119.725  1.750   
4  Alipur, Delhi - DPCC 2019-01-01 09:00:00  268.0  415.0  109.475  1.800   

       o3         aqi  hour_sin      hour_cos   dow_sin  dow_cos  month_sin  \
0   1.500  500.000000  0.965926  2.588190e-01  0.781831  0.62349        0.0   
1   1.300  500.000000  1.00000

In [ ]:
print("=" * 70)
print("STEP 3B: STATION COLUMNS")
print("=" * 70)

# Safe to re-run: drop old one-hot columns before recreating them
existing_station_cols = [
    c for c in df.columns
    if c.startswith("station_")
]

if existing_station_cols:
    df = df.drop(columns=existing_station_cols)

station_dummies = pd.get_dummies(
    df["station"],
    prefix="station",
    dtype=np.float32
)

STATION_COLS = sorted(station_dummies.columns)

df = pd.concat(
    [df, station_dummies[STATION_COLS]],
    axis=1
)

del station_dummies
gc.collect()

print(f"\n[3B] Created {len(STATION_COLS)} station columns.")

print("\n[3B] First few:")

for col in STATION_COLS[:5]:
    print(f"     {col}")

print(f"     ... and {len(STATION_COLS) - 5} more")

print(
    "\n[3B] One-hot verification "
    "(first 5 rows — each row should have exactly one 1):"
)

flags = df.iloc[:5][STATION_COLS].to_numpy()

for i in range(5):
    active_col = STATION_COLS[int(flags[i].argmax())]

    print(
        f"     {df.iloc[i]['station']}  →  "
        f"{active_col}"
    )

print(
    "\n[3B] Sample columns "
    "(pollutants + time + this row's station flag):"
)

alipur_col = "station_Alipur, Delhi - DPCC"

show_cols = [
    "station",
    "timestamp",
    "pm25",
    "aqi",
    "hour_sin",
    "hour_cos",
    alipur_col
]

print(df[show_cols].head())

print(
    f"\n[3B] Shape: "
    f"{df.shape[0]:,} rows × {df.shape[1]} columns"
)

print("\nSTEP 3B DONE")
print("=" * 70)

STEP 3B: STATION COLUMNS

[3B] Created 36 station columns.

[3B] First few:
     station_Alipur, Delhi - DPCC
     station_Anand Vihar, Delhi - DPCC
     station_Ashok Vihar, Delhi - DPCC
     station_Aya Nagar, Delhi - IMD
     station_Bawana, Delhi - DPCC
     ... and 31 more

[3B] One-hot verification (first 5 rows — each row should have exactly one 1):
     Alipur, Delhi - DPCC  →  station_Alipur, Delhi - DPCC
     Alipur, Delhi - DPCC  →  station_Alipur, Delhi - DPCC
     Alipur, Delhi - DPCC  →  station_Alipur, Delhi - DPCC
     Alipur, Delhi - DPCC  →  station_Alipur, Delhi - DPCC
     Alipur, Delhi - DPCC  →  station_Alipur, Delhi - DPCC

[3B] Sample columns (pollutants + time + this row's station flag):
                station           timestamp   pm25         aqi  hour_sin  \
0  Alipur, Delhi - DPCC 2019-01-01 05:00:00  338.0  500.000000  0.965926   
1  Alipur, Delhi - DPCC 2019-01-01 06:00:00  385.0  500.000000  1.000000   
2  Alipur, Delhi - DPCC 2019-01-01 07:00:00  311.0

In [ ]:
print("=" * 70)
print("STEP 3C: FEATURE LIST")
print("=" * 70)

TARGET_COL = "aqi"

# Past AQI is included on purpose
FEATURE_COLS = (
    POLLUTANT_COLS
    + [TARGET_COL]
    + TIME_FEATURES
    + STATION_COLS
)

print(f"\n[3C] Predicting : {TARGET_COL}")
print(f"[3C] Using      : {len(FEATURE_COLS)} features")

print(
    f"     {len(POLLUTANT_COLS)}  pollutants: "
    f"{POLLUTANT_COLS}"
)

print("     1  past AQI")

print(
    f"     {len(TIME_FEATURES)}  time-of-day / week / year"
)

print(
    f"     {len(STATION_COLS)} station identity"
)

print("\n[3C] Sample of the finished table:")

display(
    df[
        ["station", "timestamp", "aqi"]
        + POLLUTANT_COLS
    ].head(3)
)

missing = [
    c for c in FEATURE_COLS
    if c not in df.columns
]

if missing:
    raise RuntimeError(
        f"These features are not in the table: {missing}"
    )

print("\nSTEP 3C DONE")
print("=" * 70)

STEP 3C: FEATURE LIST

[3C] Predicting : aqi
[3C] Using      : 48 features
     5  pollutants: ['pm25', 'pm10', 'no2', 'co', 'o3']
     1  past AQI
     6  time-of-day / week / year
     36 station identity

[3C] Sample of the finished table:


,station,timestamp,aqi,pm25,pm10,no2,co,o3
0,"Alipur, Delhi - DPCC",2019-01-01 05:00:00,500.000000,338.0,512.0,155.100,4.300,1.500
1,"Alipur, Delhi - DPCC",2019-01-01 06:00:00,500.000000,385.0,539.0,152.775,2.975,1.300
2,"Alipur, Delhi - DPCC",2019-01-01 07:00:00,484.442857,311.0,489.0,137.400,2.400,3.125



STEP 3C DONE


In [ ]:
print("=" * 70)
print("STEP 4A: SETTINGS")
print("=" * 70)

LOOKBACK = 24
HORIZONS = [1, 6, 12, 24]
MAX_HORIZON = max(HORIZONS)
STRIDE = 6
RANDOM_SEED = 42

# Fixing the seeds means re-running gives the same result
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print(
    f"\n[4A] One sample = {LOOKBACK} hours in  "
    f"-> {len(HORIZONS)} numbers out"
)

print(f"     history  : {LOOKBACK} hours")
print(f"     predict  : {HORIZONS} hours ahead")
print(f"     stride   : {STRIDE} hours between windows")
print(f"     seed     : {RANDOM_SEED}")

print(
    f"\n[4A] Each window spans "
    f"{LOOKBACK + MAX_HORIZON} hours in total"
)

print(
    f"     ({LOOKBACK} of history plus "
    f"{MAX_HORIZON} to reach the furthest target)."
)

print("\nSTEP 4A DONE")
print("=" * 70)

STEP 4A: SETTINGS

[4A] One sample = 24 hours in  -> 4 numbers out
     history  : 24 hours
     predict  : [1, 6, 12, 24] hours ahead
     stride   : 6 hours between windows
     seed     : 42

[4A] Each window spans 48 hours in total
     (24 of history plus 24 to reach the furthest target).

STEP 4A DONE


In [ ]:
print("=" * 70)
print("STEP 4B: SPLIT DATES")
print("=" * 70)

TRAIN_END = pd.Timestamp("2022-12-31 23:59:59")
VAL_START = pd.Timestamp("2023-01-01")
VAL_END = pd.Timestamp("2023-06-30 23:59:59")
TEST_START = pd.Timestamp("2023-07-01")

data_start = df["timestamp"].min()
data_end = df["timestamp"].max()


def assign_split(when):
    """Which split does a window ending at this time belong to?"""

    if when <= TRAIN_END:
        return "train"

    if VAL_START <= when <= VAL_END:
        return "val"

    return "test"


print(
    f"\n[4B] Data covers "
    f"{data_start}  ->  {data_end}"
)

print(
    "\n[4B] Splits (decided by the LAST hour of each window):"
)

print(
    f"     TRAIN : {data_start.date()}  ..  "
    f"{TRAIN_END.date()}"
)

print(
    f"     VAL   : {VAL_START.date()}  ..  "
    f"{VAL_END.date()}"
)

print(
    f"     TEST  : {TEST_START.date()}  ..  "
    f"{data_end.date()}"
)


rows_per_split = {
    "train": int(
        (df["timestamp"] <= TRAIN_END).sum()
    ),

    "val": int(
        (
            (df["timestamp"] >= VAL_START)
            & (df["timestamp"] <= VAL_END)
        ).sum()
    ),

    "test": int(
        (df["timestamp"] >= TEST_START).sum()
    ),
}


print("\n[4B] Rows available in each split:")

for name, count in rows_per_split.items():

    print(
        f"     {name:6s} {count:10,}"
    )

    if count == 0:
        raise RuntimeError(
            f"Split '{name}' has no rows — "
            f"check the dates above."
        )


print("\nSTEP 4B DONE")
print("=" * 70)

STEP 4B: SPLIT DATES

[4B] Data covers 2019-01-01 05:00:00  ->  2024-01-01 05:00:00

[4B] Splits (decided by the LAST hour of each window):
     TRAIN : 2019-01-01  ..  2022-12-31
     VAL   : 2023-01-01  ..  2023-06-30
     TEST  : 2023-07-01  ..  2024-01-01

[4B] Rows available in each split:
     train   1,262,124
     val       156,384
     test      159,192

STEP 4B DONE


In [ ]:
print("=" * 70)
print("STEP 4C: SCALE FEATURES")
print("=" * 70)

train_rows = df["timestamp"] <= TRAIN_END

complete = df[FEATURE_COLS].notna().all(axis=1)

fit_mask = train_rows & complete

scaler = MinMaxScaler()

scaler.fit(
    df.loc[fit_mask, FEATURE_COLS]
)

print(
    f"\n[4C] Scaler fitted on "
    f"{int(fit_mask.sum()):,} complete training rows."
)


FEATURES_SCALED = np.full(
    (len(df), len(FEATURE_COLS)),
    np.nan,
    dtype=np.float32
)

FEATURES_SCALED[complete.to_numpy()] = (
    scaler.transform(
        df.loc[complete, FEATURE_COLS]
    ).astype(np.float32)
)

gc.collect()


print(
    f"[4C] Scaled array shape: "
    f"{FEATURES_SCALED.shape}"
)

print(
    f"     complete rows scaled : "
    f"{int(complete.sum()):,}"
)

print(
    f"     blank rows left as NaN: "
    f"{int((~complete).sum()):,}"
)


aqi_raw = df.loc[
    complete,
    TARGET_COL
].to_numpy()

aqi_scaled = FEATURES_SCALED[
    complete.to_numpy(),
    FEATURE_COLS.index(TARGET_COL)
]


print(
    "\n[4C] What scaling did to the AQI column "
    "(complete rows only):"
)

print(
    f"     before: "
    f"{aqi_raw.min():7.1f} .. "
    f"{aqi_raw.max():7.1f}"
)

print(
    f"     after : "
    f"{aqi_scaled.min():7.3f} .. "
    f"{aqi_scaled.max():7.3f}"
)


print("\nSTEP 4C DONE")
print("=" * 70)

STEP 4C: SCALE FEATURES

[4C] Scaler fitted on 1,100,530 complete training rows.
[4C] Scaled array shape: (1577700, 48)
     complete rows scaled : 1,388,998
     blank rows left as NaN: 188,702

[4C] What scaling did to the AQI column (complete rows only):
     before:     6.2 ..   500.0
     after :   0.000 ..   1.000

STEP 4C DONE


In [ ]:
print("=" * 70)
print("STEP 4D: DEFINE THE WINDOW BUILDER")
print("=" * 70)

station_of_row = df["station"].to_numpy()
aqi_of_row = df[TARGET_COL].to_numpy(dtype=np.float32)
time_of_row = df["timestamp"].to_numpy()


def build_windows_for_station(name):
    """
    Return (X, y, end_times, last_aqi) for one station.

    X         -- (n_windows, 24, n_features) the history
    y         -- (n_windows, 4) AQI at +1h, +6h, +12h, +24h
    end_times -- when each window ends
    last_aqi  -- AQI at the final history hour
    """

    rows = np.where(station_of_row == name)[0]

    aqi = aqi_of_row[rows]
    times = time_of_row[rows]

    X = []
    y = []
    end_times = []
    last_aqi = []

    last_possible_start = (
        len(rows) - LOOKBACK - MAX_HORIZON
    )

    for start in range(
        0,
        last_possible_start + 1,
        STRIDE
    ):

        end = start + LOOKBACK - 1

        # History features must be complete
        history = FEATURES_SCALED[
            rows[start:end + 1]
        ]

        if np.isnan(history).any():
            continue

        # Future AQI targets must also be real
        targets = np.array(
            [
                aqi[end + h]
                for h in HORIZONS
            ],
            dtype=np.float32
        )

        if np.isnan(targets).any():
            continue

        X.append(history)
        y.append(targets)
        end_times.append(times[end])
        last_aqi.append(aqi[end])

    if not X:
        return None

    return (
        np.stack(X).astype(np.float32),
        np.stack(y),
        np.array(end_times),
        np.array(last_aqi, dtype=np.float32),
    )


print(f"\n[4D] Function ready.")

print(
    f"     Each sample: "
    f"({LOOKBACK}, {len(FEATURE_COLS)})"
    f"  ->  {len(HORIZONS)} AQI values"
)

print("\nSTEP 4D DONE")
print("=" * 70)

STEP 4D: DEFINE THE WINDOW BUILDER

[4D] Function ready.
     Each sample: (24, 48)  ->  4 AQI values

STEP 4D DONE


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import gc

DRIVE_ROOT = Path("/content/drive/MyDrive/vayupath")
DATA_DIR = DRIVE_ROOT / "data"
MODEL_DIR = DRIVE_ROOT / "models"

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Setup ready.")
print("DATA_DIR :", DATA_DIR)
print("MODEL_DIR:", MODEL_DIR)

Setup ready.
DATA_DIR : /content/drive/MyDrive/vayupath/data
MODEL_DIR: /content/drive/MyDrive/vayupath/models


In [ ]:
print("=" * 70)
print("STEP 1A: DOWNLOAD YEARLY CPCB FILES (DELHI ONLY)")
print("=" * 70)

import urllib.request

YEARS = [2019, 2020, 2021, 2022, 2023]

RAW_COLS = [
    "Station Name",
    "Timestamp",
    "PM2.5 (µg/m³)",
    "PM10 (µg/m³)",
    "NO2 (µg/m³)",
    "CO (mg/m³)",
    "Ozone (µg/m³)",
]

TMP_DIR = Path("/content/cpcb_tmp")
TMP_DIR.mkdir(parents=True, exist_ok=True)

delhi_year_files = []

for year in YEARS:

    cache = DATA_DIR / f"delhi_pollutants_{year}.parquet"
    delhi_year_files.append(cache)

    if cache.exists():
        print(
            f"\n[1A] {year}: already cached -> "
            f"{cache.name} "
            f"({cache.stat().st_size / 1e6:.1f} MB)"
        )
        continue

    url = (
        f"https://github.com/Vonter/india-cpcb-aqi/releases/download/"
        f"{year}/cpcb-air-quality-{year}.parquet"
    )

    tmp = TMP_DIR / f"cpcb-air-quality-{year}.parquet"

    print(f"\n[1A] {year}: downloading...")
    urllib.request.urlretrieve(url, tmp)

    print(f"[1A] {year}: reading and filtering to Delhi...")

    year_df = pd.read_parquet(
        tmp,
        columns=["City"] + RAW_COLS
    )

    delhi = year_df[
        year_df["City"].astype(str).str.strip().str.lower() == "delhi"
    ].copy()

    delhi = delhi.drop(columns=["City"])
    delhi.to_parquet(cache, index=False)

    print(
        f"[1A] {year}: saved {len(delhi):,} Delhi rows "
        f"-> {cache.name}"
    )

    del year_df, delhi
    tmp.unlink(missing_ok=True)
    gc.collect()


print("\n[1A] Delhi cache files ready:")

for path in delhi_year_files:
    print(
        f"     {path.name:32s} "
        f"{path.stat().st_size / 1e6:6.1f} MB"
    )

print("\nSTEP 1A DONE")
print("=" * 70)

STEP 1A: DOWNLOAD YEARLY CPCB FILES (DELHI ONLY)

[1A] 2019: downloading...
[1A] 2019: reading and filtering to Delhi...
[1A] 2019: saved 1,363,514 Delhi rows -> delhi_pollutants_2019.parquet

[1A] 2020: downloading...
[1A] 2020: reading and filtering to Delhi...
[1A] 2020: saved 1,367,306 Delhi rows -> delhi_pollutants_2020.parquet

[1A] 2021: downloading...
[1A] 2021: reading and filtering to Delhi...
[1A] 2021: saved 1,363,334 Delhi rows -> delhi_pollutants_2021.parquet

[1A] 2022: downloading...
[1A] 2022: reading and filtering to Delhi...
[1A] 2022: saved 1,364,216 Delhi rows -> delhi_pollutants_2022.parquet

[1A] 2023: downloading...
[1A] 2023: reading and filtering to Delhi...
[1A] 2023: saved 1,362,463 Delhi rows -> delhi_pollutants_2023.parquet

[1A] Delhi cache files ready:
     delhi_pollutants_2019.parquet      12.6 MB
     delhi_pollutants_2020.parquet      12.4 MB
     delhi_pollutants_2021.parquet      12.7 MB
     delhi_pollutants_2022.parquet      13.0 MB
     delhi_po

In [ ]:
print("=" * 70)
print("STEP 1B: COMBINE DELHI YEARS")
print("=" * 70)

RENAME = {
    "Station Name": "station",
    "Timestamp": "timestamp",
    "PM2.5 (µg/m³)": "pm25",
    "PM10 (µg/m³)": "pm10",
    "NO2 (µg/m³)": "no2",
    "CO (mg/m³)": "co",
    "Ozone (µg/m³)": "o3",
}

POLLUTANT_COLS = [
    "pm25",
    "pm10",
    "no2",
    "co",
    "o3"
]

parts = []

for path in delhi_year_files:
    part = pd.read_parquet(path).rename(columns=RENAME)
    print(f"[1B] {path.name}: {len(part):,} rows")
    parts.append(part)

df = pd.concat(parts, ignore_index=True)

del parts
gc.collect()

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    utc=True
)

df["timestamp"] = (
    df["timestamp"]
    .dt.tz_convert("Asia/Kolkata")
    .dt.tz_localize(None)
)

df = (
    df.sort_values(["station", "timestamp"])
      .reset_index(drop=True)
)

print(
    f"\n[1B] Combined: {len(df):,} rows "
    "(still 15-minute resolution)"
)

print(f"     Stations : {df['station'].nunique()}")

print(
    f"     Range    : "
    f"{df['timestamp'].min()}  ->  "
    f"{df['timestamp'].max()}"
)

print("\n[1B] Sample:")
display(df.head(3))

print("\nSTEP 1B DONE")
print("=" * 70)

STEP 1B: COMBINE DELHI YEARS
[1B] delhi_pollutants_2019.parquet: 1,363,514 rows
[1B] delhi_pollutants_2020.parquet: 1,367,306 rows
[1B] delhi_pollutants_2021.parquet: 1,363,334 rows
[1B] delhi_pollutants_2022.parquet: 1,364,216 rows
[1B] delhi_pollutants_2023.parquet: 1,362,463 rows

[1B] Combined: 6,820,833 rows (still 15-minute resolution)
     Stations : 39
     Range    : 2019-01-01 05:30:00  ->  2024-01-01 05:15:00

[1B] Sample:


,station,timestamp,pm25,pm10,no2,co,o3
0,"Alipur, Delhi - DPCC",2019-01-01 05:30:00,338.0,512.0,145.1,3.8,1.2
1,"Alipur, Delhi - DPCC",2019-01-01 05:45:00,338.0,512.0,165.1,4.8,1.8
2,"Alipur, Delhi - DPCC",2019-01-01 06:00:00,385.0,539.0,170.0,3.6,1.3



STEP 1B DONE


In [ ]:
print("=" * 70)
print("STEP 1C: AGGREGATE TO HOURLY")
print("=" * 70)

before = len(df)

df["timestamp"] = df["timestamp"].dt.floor("h")

df = (
    df.groupby(
        ["station", "timestamp"],
        as_index=False
    )[POLLUTANT_COLS]
    .mean()
    .sort_values(["station", "timestamp"])
    .reset_index(drop=True)
)

print(
    f"\n[1C] {before:,} fifteen-minute rows -> "
    f"{len(df):,} hourly rows"
)

print(f"[1C] Stations: {df['station'].nunique()}")

print("\n[1C] Sample hourly rows:")
display(df.head(5))

print("\nSTEP 1C DONE")
print("=" * 70)

STEP 1C: AGGREGATE TO HOURLY

[1C] 6,820,833 fifteen-minute rows -> 1,707,496 hourly rows
[1C] Stations: 39

[1C] Sample hourly rows:


,station,timestamp,pm25,pm10,no2,co,o3
0,"Alipur, Delhi - DPCC",2019-01-01 05:00:00,338.0,512.0,155.100,4.300,1.500
1,"Alipur, Delhi - DPCC",2019-01-01 06:00:00,385.0,539.0,152.775,2.975,1.300
2,"Alipur, Delhi - DPCC",2019-01-01 07:00:00,311.0,489.0,137.400,2.400,3.125
3,"Alipur, Delhi - DPCC",2019-01-01 08:00:00,285.0,444.0,119.725,1.750,9.400
4,"Alipur, Delhi - DPCC",2019-01-01 09:00:00,268.0,415.0,109.475,1.800,10.150



STEP 1C DONE


In [ ]:
print("=" * 70)
print("STEP 1D: INSPECT")
print("=" * 70)

print(f"\n[1D] Rows        : {len(df):,}")
print(f"     Stations    : {df['station'].nunique()}")
print(
    f"     Date range  : "
    f"{df['timestamp'].min()} -> {df['timestamp'].max()}"
)

print("\n[1D] Missing % per pollutant:")

for col in POLLUTANT_COLS:
    pct = df[col].isna().mean() * 100
    print(f"     {col:6s}  {pct:5.1f}%")

print("\n[1D] Value ranges:")
print(f"     {'col':6s} {'min':>10} {'median':>10} {'max':>10}")

for col in POLLUTANT_COLS:
    s = df[col].dropna()
    print(
        f"     {col:6s} "
        f"{s.min():10.1f} "
        f"{s.median():10.1f} "
        f"{s.max():10.1f}"
    )

print("\n[1D] Rows per year:")

for year, count in (
    df.groupby(df["timestamp"].dt.year).size().items()
):
    print(f"     {year}: {count:,}")

print("\n[1D] Stations:")

for name in sorted(df["station"].unique()):
    print(f"     - {name}")

print("\nSTEP 1D DONE")
print("=" * 70)

STEP 1D: INSPECT

[1D] Rows        : 1,707,496
     Stations    : 39
     Date range  : 2019-01-01 05:00:00 -> 2024-01-01 05:00:00

[1D] Missing % per pollutant:
     pm25      9.4%
     pm10     12.1%
     no2       9.8%
     co       10.2%
     o3       10.5%

[1D] Value ranges:
     col           min     median        max
     pm25          0.0       69.2     1000.0
     pm10          0.0      171.0     1000.0
     no2           0.0       28.8      499.8
     co            0.0        1.0       47.8
     o3            0.0       18.2      200.0

[1D] Rows per year:
     2019: 341,030
     2020: 342,389
     2021: 341,248
     2022: 341,492
     2023: 341,103
     2024: 234

[1D] Stations:
     - Alipur, Delhi - DPCC
     - Anand Vihar, Delhi - DPCC
     - Ashok Vihar, Delhi - DPCC
     - Aya Nagar, Delhi - IMD
     - Bawana, Delhi - DPCC
     - Burari Crossing, Delhi - IMD
     - CRRI Mathura Road, Delhi - IMD
     - Chandni Chowk, Delhi - IITM
     - DTU, Delhi - CPCB
     - Dr. Karn

In [ ]:
print("=" * 70)
print("STEP 2A: CLEAN INVALID VALUES")
print("=" * 70)

LIMITS = {
    "pm25": (0, 1000),
    "pm10": (0, 1500),
    "no2": (0, 700),
    "co": (0, 50),
    "o3": (0, 500),
}

print("\n[2A] Cleaning each pollutant...")

for col in POLLUTANT_COLS:
    before_na = int(df[col].isna().sum())

    df.loc[df[col] == 999, col] = np.nan

    low, high = LIMITS[col]
    bad = df[col].notna() & ~df[col].between(low, high)

    df.loc[bad, col] = np.nan

    after_na = int(df[col].isna().sum())

    print(
        f"     {col:6s}: removed "
        f"{after_na - before_na:,} bad values "
        f"(now {df[col].isna().mean() * 100:4.1f}% blank)"
    )

before = len(df)
df = df[df["pm25"].notna()].copy()

print(f"\n[2A] Dropped rows with no PM2.5: {before - len(df):,}")
print(f"[2A] Rows left: {len(df):,}")

print("\nSTEP 2A DONE")
print("=" * 70)

STEP 2A: CLEAN INVALID VALUES

[2A] Cleaning each pollutant...
     pm25  : removed 0 bad values (now  9.4% blank)
     pm10  : removed 2 bad values (now 12.1% blank)
     no2   : removed 0 bad values (now  9.8% blank)
     co    : removed 0 bad values (now 10.2% blank)
     o3    : removed 0 bad values (now 10.5% blank)

[2A] Dropped rows with no PM2.5: 160,226
[2A] Rows left: 1,547,270

STEP 2A DONE


In [ ]:
print("=" * 70)
print("STEP 2B: STATION COVERAGE")
print("=" * 70)

MIN_COVERAGE = 0.50

date_min = df["timestamp"].min()
date_max = df["timestamp"].max()

total_hours = len(
    pd.date_range(date_min, date_max, freq="h")
)

readings = df.groupby("station")["pm25"].count()
coverage = readings / total_hours

print(f"\n[2B] Date range: {date_min}  ->  {date_max}")
print(f"[2B] Hours in the date range: {total_hours:,}")

print("\n[2B] Coverage per station:")

for name, pct in coverage.sort_values(ascending=False).items():
    flag = "keep" if pct >= MIN_COVERAGE else "DROP"
    print(
        f"     {name[:46]:46} "
        f"{pct * 100:5.1f}%   {flag}"
    )

good_stations = sorted(
    coverage[coverage >= MIN_COVERAGE].index
)

df = df[df["station"].isin(good_stations)].copy()

print(
    f"\n[2B] Keeping {len(good_stations)} "
    f"of {len(coverage)} stations."
)

print(f"[2B] Rows now: {len(df):,}")

if len(good_stations) < 5:
    raise RuntimeError(
        "Fewer than 5 stations survived. Check Step 2A."
    )

print("\nSTEP 2B DONE")
print("=" * 70)

STEP 2B: STATION COVERAGE

[2B] Date range: 2019-01-01 05:00:00  ->  2024-01-01 05:00:00
[2B] Hours in the date range: 43,825

[2B] Coverage per station:
     Major Dhyan Chand National Stadium, Delhi - DP  98.0%   keep
     Dwarka-Sector 8, Delhi - DPCC                   98.0%   keep
     Rohini, Delhi - DPCC                            97.9%   keep
     Sri Aurobindo Marg, Delhi - DPCC                97.9%   keep
     Okhla Phase-2, Delhi - DPCC                     97.7%   keep
     NSIT Dwarka, Delhi - CPCB                       97.6%   keep
     Nehru Nagar, Delhi - DPCC                       97.4%   keep
     Bawana, Delhi - DPCC                            97.1%   keep
     Shadipur, Delhi - CPCB                          97.0%   keep
     CRRI Mathura Road, Delhi - IMD                  96.8%   keep
     Sonia Vihar, Delhi - DPCC                       96.5%   keep
     Patparganj, Delhi - DPCC                        96.5%   keep
     Jawaharlal Nehru Stadium, Delhi - DPCC          9

In [ ]:
print("=" * 70)
print("STEP 2C: COMPLETE HOURLY TIMELINE")
print("=" * 70)

FFILL_LIMIT = 3

all_hours = pd.date_range(
    df["timestamp"].min(),
    df["timestamp"].max(),
    freq="h"
)

full_index = pd.MultiIndex.from_product(
    [good_stations, all_hours],
    names=["station", "timestamp"]
)

print(
    f"\n[2C] {len(good_stations)} stations x "
    f"{len(all_hours):,} hours "
    f"= {len(full_index):,} slots"
)

df = (
    df.set_index(["station", "timestamp"])
      .reindex(full_index)
      .reset_index()
)

print("\n[2C] Forward-filling gaps of up to 3 hours...")

for col in POLLUTANT_COLS:
    before = int(df[col].isna().sum())

    df[col] = (
        df.groupby("station")[col]
          .ffill(limit=FFILL_LIMIT)
    )

    after = int(df[col].isna().sum())

    print(
        f"     {col:6s}: filled {before - after:,} "
        f"| still blank {after:,}"
    )

print(
    f"\n[2C] Records now: {len(df):,} rows "
    f"({df['station'].nunique()} stations x "
    f"{df['timestamp'].nunique():,} hours)"
)

gc.collect()

print("\nSTEP 2C DONE")
print("=" * 70)

STEP 2C: COMPLETE HOURLY TIMELINE

[2C] 36 stations x 43,825 hours = 1,577,700 slots

[2C] Forward-filling gaps of up to 3 hours...
     pm25  : filled 22,777 | still blank 58,571
     pm10  : filled 27,817 | still blank 122,675
     no2   : filled 32,314 | still blank 84,938
     co    : filled 42,364 | still blank 85,881
     o3    : filled 39,018 | still blank 92,089

[2C] Records now: 1,577,700 rows (36 stations x 43,825 hours)

STEP 2C DONE


In [ ]:
print("=" * 70)
print("STEP 2D: COMPUTE CPCB AQI")
print("=" * 70)


def sub_index_array(values, breakpoints):
    values = np.asarray(values, dtype=np.float64)

    out = np.full(
        values.shape,
        np.nan,
        dtype=np.float64
    )

    valid = np.isfinite(values) & (values >= 0)

    for c_lo, c_hi, a_lo, a_hi in breakpoints:
        mask = (
            valid
            & np.isnan(out)
            & (values >= c_lo)
            & (values <= c_hi)
        )

        out[mask] = (
            a_lo
            + (a_hi - a_lo)
            * (values[mask] - c_lo)
            / (c_hi - c_lo)
        )

    top = breakpoints[-1][1]
    above = valid & (values > top)
    out[above] = 500.0

    return out


BP_PM25 = [
    (0, 30, 0, 50),
    (30, 60, 51, 100),
    (60, 90, 101, 200),
    (90, 120, 201, 300),
    (120, 250, 301, 400),
    (250, 380, 401, 500),
]

BP_PM10 = [
    (0, 50, 0, 50),
    (50, 100, 51, 100),
    (100, 250, 101, 200),
    (250, 350, 201, 300),
    (350, 430, 301, 400),
    (430, 500, 401, 500),
]

BP_NO2 = [
    (0, 40, 0, 50),
    (40, 80, 51, 100),
    (80, 180, 101, 200),
    (180, 280, 201, 300),
    (280, 400, 301, 400),
    (400, 1000, 401, 500),
]

BP_CO = [
    (0, 1.0, 0, 50),
    (1.0, 2.0, 51, 100),
    (2.0, 10.0, 101, 200),
    (10.0, 17.0, 201, 300),
    (17.0, 34.0, 301, 400),
    (34.0, 50.0, 401, 500),
]

BP_O3 = [
    (0, 50, 0, 50),
    (50, 100, 51, 100),
    (100, 168, 101, 200),
    (168, 208, 201, 300),
    (208, 748, 301, 400),
    (748, 1000, 401, 500),
]


print("\n[2D] Computing sub-indices...")

si_pm25 = sub_index_array(df["pm25"], BP_PM25)
si_pm10 = sub_index_array(df["pm10"], BP_PM10)
si_no2 = sub_index_array(df["no2"], BP_NO2)
si_co = sub_index_array(df["co"], BP_CO)
si_o3 = sub_index_array(df["o3"], BP_O3)

df["aqi"] = np.nanmax(
    np.column_stack([
        si_pm25,
        si_pm10,
        si_no2,
        si_co,
        si_o3
    ]),
    axis=1
)

df.loc[np.isnan(si_pm25), "aqi"] = np.nan
df["aqi"] = df["aqi"].clip(upper=500)

print(
    f"[2D] Rows with AQI : "
    f"{df['aqi'].notna().sum():,} / {len(df):,}"
)

print(f"     unique values : {df['aqi'].nunique():,}")

print(
    f"     mean / std    : "
    f"{df['aqi'].mean():.1f} / "
    f"{df['aqi'].std():.1f}"
)

print(
    f"     min / max     : "
    f"{df['aqi'].min():.1f} / "
    f"{df['aqi'].max():.1f}"
)

hourly_path = DATA_DIR / "delhi_cpcb_hourly.csv"

save_cols = [
    "station",
    "timestamp"
] + POLLUTANT_COLS + ["aqi"]

df.loc[
    df["aqi"].notna(),
    save_cols
].to_csv(
    hourly_path,
    index=False
)

print(f"\n[2D] Saved cleaned table -> {hourly_path}")

print("\n[2D] Sample rows:")
display(
    df.loc[
        df["aqi"].notna(),
        save_cols
    ].head(5)
)

print("\nSTEP 2D DONE")
print("=" * 70)

STEP 2D: COMPUTE CPCB AQI

[2D] Computing sub-indices...


/tmp/ipykernel_2033/2202421282.py:93: RuntimeWarning: All-NaN slice encountered
  df["aqi"] = np.nanmax(


[2D] Rows with AQI : 1,519,129 / 1,577,700
     unique values : 182,391
     mean / std    : 214.6 / 133.1
     min / max     : 1.7 / 500.0

[2D] Saved cleaned table -> /content/drive/MyDrive/vayupath/data/delhi_cpcb_hourly.csv

[2D] Sample rows:


,station,timestamp,pm25,pm10,no2,co,o3,aqi
0,"Alipur, Delhi - DPCC",2019-01-01 05:00:00,338.0,512.0,155.100,4.300,1.500,500.000000
1,"Alipur, Delhi - DPCC",2019-01-01 06:00:00,385.0,539.0,152.775,2.975,1.300,500.000000
2,"Alipur, Delhi - DPCC",2019-01-01 07:00:00,311.0,489.0,137.400,2.400,3.125,484.442857
3,"Alipur, Delhi - DPCC",2019-01-01 08:00:00,285.0,444.0,119.725,1.750,9.400,427.653846
4,"Alipur, Delhi - DPCC",2019-01-01 09:00:00,268.0,415.0,109.475,1.800,10.150,414.707692



STEP 2D DONE


In [ ]:
print("=" * 70)
print("STEP 2E: SPATIAL VARIATION CHECK")
print("=" * 70)

per_hour = (
    df.dropna(subset=["aqi"])
      .groupby("timestamp")["aqi"]
)

spread = (per_hour.max() - per_hour.min()).mean()
station_std = per_hour.std().mean()

print("\n[2E] Averaged over every hour:")
print(
    f"     gap between dirtiest and cleanest station : "
    f"{spread:.0f} AQI points"
)
print(
    f"     spread across stations (std)              : "
    f"{station_std:.0f} AQI points"
)

example_hour = (
    df.dropna(subset=["aqi"])
      .groupby("timestamp")
      .size()
      .idxmax()
)

snapshot = (
    df[df["timestamp"] == example_hour]
      .dropna(subset=["aqi"])
      .sort_values("aqi", ascending=False)
)

print(
    f"\n[2E] One real hour ({example_hour}) — "
    f"dirtiest and cleanest:"
)

for _, row in pd.concat(
    [snapshot.head(4), snapshot.tail(4)]
).iterrows():
    print(
        f"     {row['station'][:46]:46} "
        f"{row['aqi']:5.0f}"
    )

print("\n[2E] Average AQI per station:")

for name, value in (
    df.groupby("station")["aqi"]
      .mean()
      .sort_values(ascending=False)
      .items()
):
    print(
        f"     {name[:46]:46} "
        f"{value:5.0f}"
    )

if station_std < 5:
    raise RuntimeError(
        "Stations barely differ from each other. "
        "Route optimization needs real spatial variation."
    )

print(
    "\n     OK: stations genuinely differ. "
    "Route comparison is possible."
)

print("\nSTEP 2E DONE")
print("=" * 70)

STEP 2E: SPATIAL VARIATION CHECK

[2E] Averaged over every hour:
     gap between dirtiest and cleanest station : 229 AQI points
     spread across stations (std)              : 52 AQI points

[2E] One real hour (2019-01-01 05:00:00) — dirtiest and cleanest:
     Alipur, Delhi - DPCC                             500
     Anand Vihar, Delhi - DPCC                        500
     Ashok Vihar, Delhi - DPCC                        500
     CRRI Mathura Road, Delhi - IMD                   500
     Narela, Delhi - DPCC                             422
     Sri Aurobindo Marg, Delhi - DPCC                 379
     Aya Nagar, Delhi - IMD                           373
     Najafgarh, Delhi - DPCC                          348

[2E] Average AQI per station:
     Anand Vihar, Delhi - DPCC                        264
     Mundka, Delhi - DPCC                             255
     Bawana, Delhi - DPCC                             244
     Jahangirpuri, Delhi - DPCC                       244
     Wazirpur,

In [ ]:
print("=" * 70)
print("STEP 3A: TIME FEATURES")
print("=" * 70)


def as_circle(values, period):
    angle = 2 * np.pi * values / period
    return np.sin(angle), np.cos(angle)


ts = df["timestamp"]

df["hour_sin"], df["hour_cos"] = as_circle(
    ts.dt.hour, 24
)

df["dow_sin"], df["dow_cos"] = as_circle(
    ts.dt.dayofweek, 7
)

df["month_sin"], df["month_cos"] = as_circle(
    ts.dt.month - 1, 12
)

TIME_FEATURES = [
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos"
]

print(
    f"\n[3A] Added {len(TIME_FEATURES)} time features: "
    f"{TIME_FEATURES}"
)

for hour in (22, 23, 0, 1):
    s, c = as_circle(np.array([hour]), 24)
    print(
        f"     {hour:02d}:00  "
        f"sin={s[0]:+.2f}  cos={c[0]:+.2f}"
    )

print("\n[3A] Dataset with time features:")
print(df.head())

print(
    f"\n[3A] Shape: "
    f"{df.shape[0]:,} rows × {df.shape[1]} columns"
)

print("\nSTEP 3A DONE")
print("=" * 70)

STEP 3A: TIME FEATURES

[3A] Added 6 time features: ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']
     22:00  sin=-0.50  cos=+0.87
     23:00  sin=-0.26  cos=+0.97
     00:00  sin=+0.00  cos=+1.00
     01:00  sin=+0.26  cos=+0.97

[3A] Dataset with time features:
                station           timestamp   pm25   pm10      no2     co  \
0  Alipur, Delhi - DPCC 2019-01-01 05:00:00  338.0  512.0  155.100  4.300   
1  Alipur, Delhi - DPCC 2019-01-01 06:00:00  385.0  539.0  152.775  2.975   
2  Alipur, Delhi - DPCC 2019-01-01 07:00:00  311.0  489.0  137.400  2.400   
3  Alipur, Delhi - DPCC 2019-01-01 08:00:00  285.0  444.0  119.725  1.750   
4  Alipur, Delhi - DPCC 2019-01-01 09:00:00  268.0  415.0  109.475  1.800   

       o3         aqi  hour_sin      hour_cos   dow_sin  dow_cos  month_sin  \
0   1.500  500.000000  0.965926  2.588190e-01  0.781831  0.62349        0.0   
1   1.300  500.000000  1.000000  6.123234e-17  0.781831  0.62349        0.0   
2   3.125

In [ ]:
print("=" * 70)
print("STEP 3B: STATION COLUMNS")
print("=" * 70)

existing_station_cols = [
    c for c in df.columns
    if c.startswith("station_")
]

if existing_station_cols:
    df = df.drop(columns=existing_station_cols)

station_dummies = pd.get_dummies(
    df["station"],
    prefix="station",
    dtype=np.float32
)

STATION_COLS = sorted(station_dummies.columns)

df = pd.concat(
    [df, station_dummies[STATION_COLS]],
    axis=1
)

del station_dummies
gc.collect()

print(f"\n[3B] Created {len(STATION_COLS)} station columns.")

print("\n[3B] First few:")
for col in STATION_COLS[:5]:
    print(f"     {col}")

print(f"     ... and {len(STATION_COLS) - 5} more")

print("\n[3B] One-hot verification:")

flags = df.iloc[:5][STATION_COLS].to_numpy()

for i in range(5):
    active_col = STATION_COLS[int(flags[i].argmax())]
    print(
        f"     {df.iloc[i]['station']}  →  {active_col}"
    )

alipur_col = "station_Alipur, Delhi - DPCC"

show_cols = [
    "station",
    "timestamp",
    "pm25",
    "aqi",
    "hour_sin",
    "hour_cos",
    alipur_col
]

print("\n[3B] Sample:")
print(df[show_cols].head())

print(
    f"\n[3B] Shape: "
    f"{df.shape[0]:,} rows × {df.shape[1]} columns"
)

print("\nSTEP 3B DONE")
print("=" * 70)

STEP 3B: STATION COLUMNS

[3B] Created 36 station columns.

[3B] First few:
     station_Alipur, Delhi - DPCC
     station_Anand Vihar, Delhi - DPCC
     station_Ashok Vihar, Delhi - DPCC
     station_Aya Nagar, Delhi - IMD
     station_Bawana, Delhi - DPCC
     ... and 31 more

[3B] One-hot verification:
     Alipur, Delhi - DPCC  →  station_Alipur, Delhi - DPCC
     Alipur, Delhi - DPCC  →  station_Alipur, Delhi - DPCC
     Alipur, Delhi - DPCC  →  station_Alipur, Delhi - DPCC
     Alipur, Delhi - DPCC  →  station_Alipur, Delhi - DPCC
     Alipur, Delhi - DPCC  →  station_Alipur, Delhi - DPCC

[3B] Sample:
                station           timestamp   pm25         aqi  hour_sin  \
0  Alipur, Delhi - DPCC 2019-01-01 05:00:00  338.0  500.000000  0.965926   
1  Alipur, Delhi - DPCC 2019-01-01 06:00:00  385.0  500.000000  1.000000   
2  Alipur, Delhi - DPCC 2019-01-01 07:00:00  311.0  484.442857  0.965926   
3  Alipur, Delhi - DPCC 2019-01-01 08:00:00  285.0  427.653846  0.866025   
4  A

In [ ]:
print("=" * 70)
print("STEP 3C: FEATURE LIST")
print("=" * 70)

TARGET_COL = "aqi"

FEATURE_COLS = (
    POLLUTANT_COLS
    + [TARGET_COL]
    + TIME_FEATURES
    + STATION_COLS
)

print(f"\n[3C] Predicting : {TARGET_COL}")
print(f"[3C] Using      : {len(FEATURE_COLS)} features")
print(f"     {len(POLLUTANT_COLS)} pollutants: {POLLUTANT_COLS}")
print("     1 past AQI")
print(f"     {len(TIME_FEATURES)} time features")
print(f"     {len(STATION_COLS)} station identity")

print("\n[3C] Sample:")
display(
    df[
        ["station", "timestamp", "aqi"]
        + POLLUTANT_COLS
    ].head(3)
)

missing = [
    c for c in FEATURE_COLS
    if c not in df.columns
]

if missing:
    raise RuntimeError(
        f"These features are not in the table: {missing}"
    )

print("\nSTEP 3C DONE")
print("=" * 70)

STEP 3C: FEATURE LIST

[3C] Predicting : aqi
[3C] Using      : 48 features
     5 pollutants: ['pm25', 'pm10', 'no2', 'co', 'o3']
     1 past AQI
     6 time features
     36 station identity

[3C] Sample:


,station,timestamp,aqi,pm25,pm10,no2,co,o3
0,"Alipur, Delhi - DPCC",2019-01-01 05:00:00,500.000000,338.0,512.0,155.100,4.300,1.500
1,"Alipur, Delhi - DPCC",2019-01-01 06:00:00,500.000000,385.0,539.0,152.775,2.975,1.300
2,"Alipur, Delhi - DPCC",2019-01-01 07:00:00,484.442857,311.0,489.0,137.400,2.400,3.125



STEP 3C DONE


In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

TensorFlow: 2.20.0


In [ ]:
print("=" * 70)
print("STEP 4B: SPLIT DATES")
print("=" * 70)

TRAIN_END = pd.Timestamp("2022-12-31 23:59:59")
VAL_START = pd.Timestamp("2023-01-01")
VAL_END = pd.Timestamp("2023-06-30 23:59:59")
TEST_START = pd.Timestamp("2023-07-01")

data_start = df["timestamp"].min()
data_end = df["timestamp"].max()


def assign_split(when):
    if when <= TRAIN_END:
        return "train"

    if VAL_START <= when <= VAL_END:
        return "val"

    return "test"


print(f"\n[4B] Data covers {data_start}  ->  {data_end}")

print("\n[4B] Splits:")
print(f"     TRAIN : {data_start.date()}  ..  {TRAIN_END.date()}")
print(f"     VAL   : {VAL_START.date()}  ..  {VAL_END.date()}")
print(f"     TEST  : {TEST_START.date()}  ..  {data_end.date()}")


rows_per_split = {
    "train": int(
        (df["timestamp"] <= TRAIN_END).sum()
    ),
    "val": int(
        (
            (df["timestamp"] >= VAL_START)
            & (df["timestamp"] <= VAL_END)
        ).sum()
    ),
    "test": int(
        (df["timestamp"] >= TEST_START).sum()
    ),
}

print("\n[4B] Rows available in each split:")

for name, count in rows_per_split.items():
    print(f"     {name:6s} {count:10,}")

    if count == 0:
        raise RuntimeError(
            f"Split '{name}' has no rows — check the dates above."
        )

print("\nSTEP 4B DONE")
print("=" * 70)

STEP 4B: SPLIT DATES

[4B] Data covers 2019-01-01 05:00:00  ->  2024-01-01 05:00:00

[4B] Splits:
     TRAIN : 2019-01-01  ..  2022-12-31
     VAL   : 2023-01-01  ..  2023-06-30
     TEST  : 2023-07-01  ..  2024-01-01

[4B] Rows available in each split:
     train   1,262,124
     val       156,384
     test      159,192

STEP 4B DONE


In [ ]:
from sklearn.preprocessing import MinMaxScaler

print("=" * 70)
print("STEP 4C: SCALE FEATURES")
print("=" * 70)

train_rows = df["timestamp"] <= TRAIN_END
complete = df[FEATURE_COLS].notna().all(axis=1)
fit_mask = train_rows & complete

scaler = MinMaxScaler()

scaler.fit(
    df.loc[fit_mask, FEATURE_COLS]
)

print(
    f"\n[4C] Scaler fitted on "
    f"{int(fit_mask.sum()):,} complete training rows."
)

FEATURES_SCALED = np.full(
    (len(df), len(FEATURE_COLS)),
    np.nan,
    dtype=np.float32
)

FEATURES_SCALED[complete.to_numpy()] = (
    scaler.transform(
        df.loc[complete, FEATURE_COLS]
    ).astype(np.float32)
)

gc.collect()

print(f"[4C] Scaled array shape: {FEATURES_SCALED.shape}")
print(f"     complete rows scaled : {int(complete.sum()):,}")
print(f"     blank rows left as NaN: {int((~complete).sum()):,}")

aqi_raw = df.loc[complete, TARGET_COL].to_numpy()

aqi_scaled = FEATURES_SCALED[
    complete.to_numpy(),
    FEATURE_COLS.index(TARGET_COL)
]

print("\n[4C] What scaling did to the AQI column:")
print(f"     before: {aqi_raw.min():7.1f} .. {aqi_raw.max():7.1f}")
print(f"     after : {aqi_scaled.min():7.3f} .. {aqi_scaled.max():7.3f}")

print("\nSTEP 4C DONE")
print("=" * 70)


STEP 4C: SCALE FEATURES

[4C] Scaler fitted on 1,100,530 complete training rows.
[4C] Scaled array shape: (1577700, 48)
     complete rows scaled : 1,388,998
     blank rows left as NaN: 188,702

[4C] What scaling did to the AQI column:
     before:     6.2 ..   500.0
     after :   0.000 ..   1.000

STEP 4C DONE


In [ ]:
print("=" * 70)
print("STEP 4D: DEFINE THE WINDOW BUILDER")
print("=" * 70)

station_of_row = df["station"].to_numpy()
aqi_of_row = df[TARGET_COL].to_numpy(dtype=np.float32)
time_of_row = df["timestamp"].to_numpy()


def build_windows_for_station(name):
    """
    Return (X, y, end_times, last_aqi) for one station.
    """

    rows = np.where(station_of_row == name)[0]

    aqi = aqi_of_row[rows]
    times = time_of_row[rows]

    X = []
    y = []
    end_times = []
    last_aqi = []

    last_possible_start = (
        len(rows) - LOOKBACK - MAX_HORIZON
    )

    for start in range(
        0,
        last_possible_start + 1,
        STRIDE
    ):
        end = start + LOOKBACK - 1

        history = FEATURES_SCALED[
            rows[start:end + 1]
        ]

        if np.isnan(history).any():
            continue

        targets = np.array(
            [aqi[end + h] for h in HORIZONS],
            dtype=np.float32
        )

        if np.isnan(targets).any():
            continue

        X.append(history)
        y.append(targets)
        end_times.append(times[end])
        last_aqi.append(aqi[end])

    if not X:
        return None

    return (
        np.stack(X).astype(np.float32),
        np.stack(y),
        np.array(end_times),
        np.array(last_aqi, dtype=np.float32),
    )


print(
    f"\n[4D] Function ready."
)

print(
    f"     Each sample: "
    f"({LOOKBACK}, {len(FEATURE_COLS)}) "
    f"-> {len(HORIZONS)} AQI values"
)

print("\nSTEP 4D DONE")
print("=" * 70)

STEP 4D: DEFINE THE WINDOW BUILDER

[4D] Function ready.
     Each sample: (24, 48) -> 4 AQI values

STEP 4D DONE


In [14]:
print(len(good_stations))
print(good_stations[:5])

NameError: name 'good_stations' is not defined

In [15]:
print("=" * 70)
print("STEP 4E: BUILD WINDOWS")
print("=" * 70)

buckets = {
    split: {
        "X": [],
        "y": [],
        "end": [],
        "last": [],
        "station": []
    }
    for split in ("train", "val", "test")
}

print("\n[4E] Station                                  train    val   test")
print("     " + "-" * 60)

for name in good_stations:

    result = build_windows_for_station(name)

    if result is None:
        print(
            f"     {name[:38]:38}  no usable windows"
        )
        continue

    X_station, y_station, ends, lasts = result

    labels = np.array([
        assign_split(pd.Timestamp(t))
        for t in ends
    ])

    counts = {}

    for split in ("train", "val", "test"):

        pick = labels == split
        counts[split] = int(pick.sum())

        if counts[split] == 0:
            continue

        buckets[split]["X"].append(
            X_station[pick]
        )

        buckets[split]["y"].append(
            y_station[pick]
        )

        buckets[split]["end"].append(
            ends[pick]
        )

        buckets[split]["last"].append(
            lasts[pick]
        )

        buckets[split]["station"].append(
            np.full(
                counts[split],
                name,
                dtype=object
            )
        )

    print(
        f"     {name[:38]:38} "
        f"{counts['train']:6,} "
        f"{counts['val']:6,} "
        f"{counts['test']:6,}"
    )

    del X_station, y_station, ends, lasts, result

gc.collect()

print("\nSTEP 4E DONE")
print("=" * 70)

STEP 4E: BUILD WINDOWS

[4E] Station                                  train    val   test
     ------------------------------------------------------------


NameError: name 'good_stations' is not defined

In [16]:
print(len(good_stations))
print(good_stations[:5])

NameError: name 'good_stations' is not defined

In [12]:

print("=" * 70)
print("STEP 4F: ASSEMBLE THE SPLITS")
print("=" * 70)


def join(split, key):
    """Concatenate one bucket, then drop the pieces."""
    pieces = buckets[split].pop(key, None)
    if not pieces:
        return None
    joined = np.concatenate(pieces, axis=0)
    pieces.clear()
    gc.collect()
    return joined


X_train, y_train = join("train", "X"), join("train", "y")
X_val, y_val = join("val", "X"), join("val", "y")
X_test, y_test = join("test", "X"), join("test", "y")

end_train, end_val, end_test = join("train", "end"), join("val", "end"), join("test", "end")
station_test = join("test", "station")
last_aqi_val, last_aqi_test = join("val", "last"), join("test", "last")

del buckets
gc.collect()

for name, array in (("train", X_train), ("val", X_val), ("test", X_test)):
    if array is None or len(array) == 0:
        raise RuntimeError(f"Split '{name}' is empty — check the dates in Step 4B.")

total = len(X_train) + len(X_val) + len(X_test)
print("\n[4F] Samples per split:")
for name, array in (("Train", X_train), ("Val", X_val), ("Test", X_test)):
    print(f"     {name:6s} {len(array):8,}  ({100 * len(array) / total:4.1f}%)")

print(f"\n[4F] X_train shape = {X_train.shape}")
print(f"     y_train shape = {y_train.shape}")
print(f"     memory used   = {(X_train.nbytes + X_val.nbytes + X_test.nbytes) / 1e6:,.0f} MB")

print("\n[4F] Dates each split actually covers:")
for name, ends in (("Train", end_train), ("Val", end_val), ("Test", end_test)):
    print(f"     {name:6s} {pd.Timestamp(ends.min()).date()}  ->  {pd.Timestamp(ends.max()).date()}")

print("\n[4F] The +1h target in each split:")
print(f"     {'split':6} {'mean':>8} {'std':>8} {'min':>7} {'max':>7}")
for name, y in (("train", y_train), ("val", y_val), ("test", y_test)):
    col = y[:, 0]
    print(f"     {name:6} {col.mean():8.1f} {col.std():8.1f} {col.min():7.0f} {col.max():7.0f}")

if y_test[:, 0].std() < 5:
    raise RuntimeError("Test targets barely vary — something is wrong with the data.")

print("\nSTEP 4F DONE")
print("=" * 70)

STEP 4F: ASSEMBLE THE SPLITS


NameError: name 'buckets' is not defined

In [ ]:
print("=" * 70)
print("STEP 5A: SCORING FUNCTION")
print("=" * 70)

def score_model(y_true, y_pred, name):
    """Print a table of accuracy per horizon and return it as a list."""
    results = []

    print(f"\n[{name}]")
    print(
        f"{'Horizon':>9} | {'RMSE':>9} | "
        f"{'MAE':>9} | {'MAPE%':>8} | {'R2':>7}"
    )
    print("-" * 56)

    for i, hours in enumerate(HORIZONS):

        actual = np.asarray(y_true)[:, i].astype(np.float64)
        predicted = np.asarray(y_pred)[:, i].astype(np.float64)

        rmse = float(
            np.sqrt(
                mean_squared_error(actual, predicted)
            )
        )

        mae = float(
            mean_absolute_error(actual, predicted)
        )

        mape = float(
            np.mean(
                np.abs(
                    (actual - predicted) / actual
                )
            ) * 100
        )

        spread = float(np.var(actual))

        r2 = float(
            1.0
            - np.mean((actual - predicted) ** 2)
            / spread
        )

        print(
            f"{'+' + str(hours) + 'h':>9} | "
            f"{rmse:9.3f} | "
            f"{mae:9.3f} | "
            f"{mape:8.2f} | "
            f"{r2:7.3f}"
        )

        results.append({
            "horizon_h": hours,
            "rmse": rmse,
            "mae": mae,
            "mape": mape,
            "r2": r2
        })

    return results


print("\n[5A] score_model() ready.")
print("     Lower RMSE is better. Higher R2 is better.")

print("\nSTEP 5A DONE")
print("=" * 70)

STEP 5A: SCORING FUNCTION

[5A] score_model() ready.
     Lower RMSE is better. Higher R2 is better.

STEP 5A DONE


In [ ]:
print("=" * 70)
print("STEP 5B: PERSISTENCE BASELINE")
print("=" * 70)

n_horizons = len(HORIZONS)

persist_val_pred = np.repeat(
    last_aqi_val[:, None],
    n_horizons,
    axis=1
)

persist_test_pred = np.repeat(
    last_aqi_test[:, None],
    n_horizons,
    axis=1
)

persist_val_metrics = score_model(
    y_val,
    persist_val_pred,
    "Persistence VAL"
)

persist_test_metrics = score_model(
    y_test,
    persist_test_pred,
    "Persistence TEST"
)

print(
    "\n[5B] Notice how the error grows with the horizon: "
    "'nothing changes'"
)

print(
    "     is a fair guess an hour ahead and a poor one a day ahead."
)

print("\nSTEP 5B DONE")
print("=" * 70)

STEP 5B: PERSISTENCE BASELINE

[Persistence VAL]
  Horizon |      RMSE |       MAE |    MAPE% |      R2
--------------------------------------------------------
      +1h |    40.747 |    24.534 |    13.87 |   0.876
      +6h |   106.960 |    76.240 |    44.76 |   0.168
     +12h |   119.175 |    88.609 |    53.67 |  -0.026
     +24h |   101.904 |    72.272 |    45.60 |   0.257

[Persistence TEST]
  Horizon |      RMSE |       MAE |    MAPE% |      R2
--------------------------------------------------------
      +1h |    33.155 |    18.422 |    10.26 |   0.945
      +6h |    88.904 |    58.521 |    32.36 |   0.614
     +12h |    96.019 |    64.838 |    37.13 |   0.550
     +24h |    71.312 |    46.203 |    27.48 |   0.753

[5B] Notice how the error grows with the horizon: 'nothing changes'
     is a fair guess an hour ahead and a poor one a day ahead.

STEP 5B DONE


In [ ]:
from sklearn.linear_model import Ridge
print("=" * 70)
print("STEP 5C: RIDGE BASELINE")
print("=" * 70)

# X is (samples, 24 hours, features) — take just the final hour.
train_last_hour = X_train[:, -1, :]
val_last_hour = X_val[:, -1, :]
test_last_hour = X_test[:, -1, :]

print(f"\n[5C] Ridge input shape: {train_last_hour.shape}  (one hour, not 24)")

ridge = Ridge(alpha=1.0)
ridge.fit(train_last_hour, y_train)
print("[5C] Trained.")

ridge_val_pred = ridge.predict(val_last_hour)
ridge_test_pred = ridge.predict(test_last_hour)

ridge_val_metrics = score_model(y_val, ridge_val_pred, "Ridge VAL")
ridge_test_metrics = score_model(y_test, ridge_test_pred, "Ridge TEST")

print("\n[5C] Prediction spread on TEST (+1h) — a healthy model's")
print("     predictions should vary about as much as reality does:")
print(f"     persistence std = {persist_test_pred[:, 0].std():8.2f}")
print(f"     ridge       std = {ridge_test_pred[:, 0].std():8.2f}")
print(f"     actual      std = {y_test[:, 0].std():8.2f}")

del train_last_hour, val_last_hour, test_last_hour
gc.collect()

print("\nSTEP 5C DONE")
print("=" * 70)

STEP 5C: RIDGE BASELINE

[5C] Ridge input shape: (153199, 48)  (one hour, not 24)
[5C] Trained.

[Ridge VAL]
  Horizon |      RMSE |       MAE |    MAPE% |      R2
--------------------------------------------------------
      +1h |    39.249 |    25.014 |    14.39 |   0.885
      +6h |    88.221 |    67.174 |    44.81 |   0.434
     +12h |    92.805 |    71.897 |    48.43 |   0.378
     +24h |    89.829 |    66.156 |    44.38 |   0.423

[Ridge TEST]
  Horizon |      RMSE |       MAE |    MAPE% |      R2
--------------------------------------------------------
      +1h |    31.681 |    19.410 |    11.36 |   0.950
      +6h |    76.653 |    57.458 |    36.86 |   0.713
     +12h |    79.240 |    60.393 |    39.75 |   0.694
     +24h |    67.596 |    48.198 |    30.80 |   0.778

[5C] Prediction spread on TEST (+1h) — a healthy model's
     predictions should vary about as much as reality does:
     persistence std =   142.74
     ridge       std =   135.49
     actual      std =   141.34

In [ ]:
print("=" * 70)
print("STEP 6A: SCALE THE TARGET")
print("=" * 70)

target_scaler = MinMaxScaler()

target_scaler.fit(y_train)

y_train_scaled = target_scaler.transform(
    y_train
).astype(np.float32)

y_val_scaled = target_scaler.transform(
    y_val
).astype(np.float32)

print(
    f"\n[6A] Target before scaling: "
    f"{y_train.min():7.1f} .. {y_train.max():7.1f}"
)

print(
    f"[6A] Target after scaling : "
    f"{y_train_scaled.min():7.3f} .. "
    f"{y_train_scaled.max():7.3f}"
)

check = target_scaler.inverse_transform(
    y_train_scaled[:100]
)

if not np.allclose(
    check,
    y_train[:100],
    atol=1e-3
):
    raise RuntimeError(
        "Target scaler round-trip failed — "
        "do not trust any metric."
    )

print(
    "[6A] Round-trip check passed "
    "(scaled -> original gives the same numbers)."
)

print("\nSTEP 6A DONE")
print("=" * 70)

STEP 6A: SCALE THE TARGET

[6A] Target before scaling:     6.7 ..   500.0
[6A] Target after scaling :   0.000 ..   1.000
[6A] Round-trip check passed (scaled -> original gives the same numbers).

STEP 6A DONE


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import gc

DATA_DIR = Path("/content/drive/MyDrive/vayupath/data")

YEARS = [2019, 2020, 2021, 2022, 2023]

delhi_year_files = [
    DATA_DIR / f"delhi_pollutants_{year}.parquet"
    for year in YEARS
]

RENAME = {
    "Station Name": "station",
    "Timestamp": "timestamp",
    "PM2.5 (µg/m³)": "pm25",
    "PM10 (µg/m³)": "pm10",
    "NO2 (µg/m³)": "no2",
    "CO (mg/m³)": "co",
    "Ozone (µg/m³)": "o3",
}

POLLUTANT_COLS = ["pm25", "pm10", "no2", "co", "o3"]

for path in delhi_year_files:
    print(path.name, "->", path.exists())

print("\nSetup ready.")

delhi_pollutants_2019.parquet -> False
delhi_pollutants_2020.parquet -> False
delhi_pollutants_2021.parquet -> False
delhi_pollutants_2022.parquet -> False
delhi_pollutants_2023.parquet -> False

Setup ready.


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/vayupath")
DATA_DIR = DRIVE_ROOT / "data"
MODEL_DIR = DRIVE_ROOT / "models"

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Setup ready.")
print("DATA_DIR :", DATA_DIR)
print("MODEL_DIR:", MODEL_DIR)

Setup ready.
DATA_DIR : /content/drive/MyDrive/vayupath/data
MODEL_DIR: /content/drive/MyDrive/vayupath/models


In [ ]:
print("=" * 70)
print("STEP 1A: DOWNLOAD YEARLY CPCB FILES (DELHI ONLY)")
print("=" * 70)

import urllib.request
from pathlib import Path
import pandas as pd
import gc

DATA_DIR = Path("/content/drive/MyDrive/vayupath/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2019, 2020, 2021, 2022, 2023]

RAW_COLS = [
    "Station Name",
    "Timestamp",
    "PM2.5 (µg/m³)",
    "PM10 (µg/m³)",
    "NO2 (µg/m³)",
    "CO (mg/m³)",
    "Ozone (µg/m³)",
]

TMP_DIR = Path("/content/cpcb_tmp")
TMP_DIR.mkdir(parents=True, exist_ok=True)

delhi_year_files = []

for year in YEARS:

    cache = DATA_DIR / f"delhi_pollutants_{year}.parquet"
    delhi_year_files.append(cache)

    if cache.exists():
        print(
            f"\n[1A] {year}: already cached -> "
            f"{cache.name} "
            f"({cache.stat().st_size / 1e6:.1f} MB)"
        )
        continue

    url = (
        f"https://github.com/Vonter/india-cpcb-aqi/releases/download/"
        f"{year}/cpcb-air-quality-{year}.parquet"
    )

    tmp = TMP_DIR / f"cpcb-air-quality-{year}.parquet"

    print(f"\n[1A] {year}: downloading...")
    print(f"     {url}")

    urllib.request.urlretrieve(url, tmp)

    print(
        f"     downloaded "
        f"{tmp.stat().st_size / 1e6:.1f} MB"
    )

    print(f"[1A] {year}: reading and filtering to Delhi...")

    year_df = pd.read_parquet(
        tmp,
        columns=["City"] + RAW_COLS
    )

    delhi = year_df[
        year_df["City"].astype(str).str.strip().str.lower()
        == "delhi"
    ].copy()

    delhi = delhi.drop(columns=["City"])

    delhi.to_parquet(cache, index=False)

    print(
        f"[1A] {year}: saved "
        f"{len(delhi):,} Delhi rows -> "
        f"{cache.name} "
        f"({cache.stat().st_size / 1e6:.1f} MB)"
    )

    del year_df, delhi
    tmp.unlink(missing_ok=True)
    gc.collect()


print("\n[1A] Delhi cache files ready:")

for path in delhi_year_files:
    print(
        f"     {path.name:32s} "
        f"{path.stat().st_size / 1e6:6.1f} MB"
    )

print("\n[1A] Checking files:")

for path in delhi_year_files:
    print(
        f"     {path.name} -> "
        f"{'OK' if path.exists() else 'MISSING'}"
    )

print("\nSTEP 1A DONE")
print("=" * 70)

STEP 1A: DOWNLOAD YEARLY CPCB FILES (DELHI ONLY)

[1A] 2019: downloading...
     https://github.com/Vonter/india-cpcb-aqi/releases/download/2019/cpcb-air-quality-2019.parquet
     downloaded 282.6 MB
[1A] 2019: reading and filtering to Delhi...
[1A] 2019: saved 1,363,514 Delhi rows -> delhi_pollutants_2019.parquet (12.6 MB)

[1A] 2020: downloading...
     https://github.com/Vonter/india-cpcb-aqi/releases/download/2020/cpcb-air-quality-2020.parquet
     downloaded 335.1 MB
[1A] 2020: reading and filtering to Delhi...
[1A] 2020: saved 1,367,306 Delhi rows -> delhi_pollutants_2020.parquet (12.4 MB)

[1A] 2021: downloading...
     https://github.com/Vonter/india-cpcb-aqi/releases/download/2021/cpcb-air-quality-2021.parquet
     downloaded 405.7 MB
[1A] 2021: reading and filtering to Delhi...
[1A] 2021: saved 1,363,334 Delhi rows -> delhi_pollutants_2021.parquet (12.7 MB)

[1A] 2022: downloading...
     https://github.com/Vonter/india-cpcb-aqi/releases/download/2022/cpcb-air-quality-2022.pa

In [ ]:
print("=" * 70)
print("STEP 1B: COMBINE DELHI YEARS")
print("=" * 70)

RENAME = {
    "Station Name": "station",
    "Timestamp": "timestamp",
    "PM2.5 (µg/m³)": "pm25",
    "PM10 (µg/m³)": "pm10",
    "NO2 (µg/m³)": "no2",
    "CO (mg/m³)": "co",
    "Ozone (µg/m³)": "o3",
}

POLLUTANT_COLS = ["pm25", "pm10", "no2", "co", "o3"]

parts = []

for path in delhi_year_files:
    part = pd.read_parquet(path).rename(columns=RENAME)

    print(
        f"[1B] {path.name}: "
        f"{len(part):,} rows"
    )

    parts.append(part)

df = pd.concat(
    parts,
    ignore_index=True
)

del parts
gc.collect()

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    utc=True
)

df["timestamp"] = (
    df["timestamp"]
    .dt.tz_convert("Asia/Kolkata")
    .dt.tz_localize(None)
)

df = (
    df.sort_values(
        ["station", "timestamp"]
    )
    .reset_index(drop=True)
)

print(
    f"\n[1B] Combined: "
    f"{len(df):,} rows"
)

print(
    f"     Stations : "
    f"{df['station'].nunique()}"
)

print(
    f"     Range    : "
    f"{df['timestamp'].min()} -> "
    f"{df['timestamp'].max()}"
)

print("\n[1B] Sample:")
display(df.head(3))

print("\nSTEP 1B DONE")
print("=" * 70)

STEP 1B: COMBINE DELHI YEARS
[1B] delhi_pollutants_2019.parquet: 1,363,514 rows
[1B] delhi_pollutants_2020.parquet: 1,367,306 rows
[1B] delhi_pollutants_2021.parquet: 1,363,334 rows
[1B] delhi_pollutants_2022.parquet: 1,364,216 rows
[1B] delhi_pollutants_2023.parquet: 1,362,463 rows

[1B] Combined: 6,820,833 rows
     Stations : 39
     Range    : 2019-01-01 05:30:00 -> 2024-01-01 05:15:00

[1B] Sample:


,station,timestamp,pm25,pm10,no2,co,o3
0,"Alipur, Delhi - DPCC",2019-01-01 05:30:00,338.0,512.0,145.1,3.8,1.2
1,"Alipur, Delhi - DPCC",2019-01-01 05:45:00,338.0,512.0,165.1,4.8,1.8
2,"Alipur, Delhi - DPCC",2019-01-01 06:00:00,385.0,539.0,170.0,3.6,1.3



STEP 1B DONE


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
print("X_train exists:", "X_train" in globals())
print("df exists:", "df" in globals())
print("buckets exists:", "buckets" in globals())

X_train exists: False
df exists: True
buckets exists: True


In [ ]:
print("=" * 70)
print("RECOVER: GOOD STATIONS")
print("=" * 70)

MIN_COVERAGE = 0.50

date_min = df["timestamp"].min()
date_max = df["timestamp"].max()

total_hours = len(
    pd.date_range(date_min, date_max, freq="h")
)

readings = df.groupby("station")["pm25"].count()

coverage = readings / total_hours

good_stations = sorted(
    coverage[coverage >= MIN_COVERAGE].index
)

print(f"\nStations available : {len(coverage)}")
print(f"Stations kept      : {len(good_stations)}")

for name in good_stations:
    print(f"  - {name}")

if len(good_stations) == 0:
    raise RuntimeError("No stations passed the 50% coverage requirement.")

print("\nGOOD STATIONS READY")

RECOVER: GOOD STATIONS

Stations available : 39
Stations kept      : 39
  - Alipur, Delhi - DPCC
  - Anand Vihar, Delhi - DPCC
  - Ashok Vihar, Delhi - DPCC
  - Aya Nagar, Delhi - IMD
  - Bawana, Delhi - DPCC
  - Burari Crossing, Delhi - IMD
  - CRRI Mathura Road, Delhi - IMD
  - Chandni Chowk, Delhi - IITM
  - DTU, Delhi - CPCB
  - Dr. Karni Singh Shooting Range, Delhi - DPCC
  - Dwarka-Sector 8, Delhi - DPCC 
  - IGI Airport (T3), Delhi - IMD
  - IHBAS, Dilshad Garden, Delhi - CPCB
  - ITO, Delhi - CPCB
  - Jahangirpuri, Delhi - DPCC
  - Jawaharlal Nehru Stadium, Delhi - DPCC
  - Lodhi Road, Delhi - IITM
  - Lodhi Road, Delhi - IMD
  - Major Dhyan Chand National Stadium, Delhi - DPCC
  - Mandir Marg, Delhi - DPCC
  - Mundka, Delhi - DPCC
  - NSIT Dwarka, Delhi - CPCB
  - Najafgarh, Delhi - DPCC
  - Narela, Delhi - DPCC
  - Nehru Nagar, Delhi - DPCC
  - North Campus, DU, Delhi - IMD
  - Okhla Phase-2, Delhi - DPCC
  - Patparganj, Delhi - DPCC
  - Punjabi Bagh, Delhi - DPCC
  - Pusa, D

In [ ]:
print("=" * 70)
print("STEP 2A: CLEAN INVALID VALUES")
print("=" * 70)

LIMITS = {
    "pm25": (0, 1000),
    "pm10": (0, 1500),
    "no2": (0, 700),
    "co": (0, 50),
    "o3": (0, 500),
}

for col in POLLUTANT_COLS:
    before_na = int(df[col].isna().sum())

    df.loc[df[col] == 999, col] = np.nan

    low, high = LIMITS[col]

    bad = (
        df[col].notna()
        & ~df[col].between(low, high)
    )

    df.loc[bad, col] = np.nan

    after_na = int(df[col].isna().sum())

    print(
        f"     {col:6s}: removed "
        f"{after_na - before_na:,} bad values"
    )

before = len(df)

df = df[df["pm25"].notna()].copy()

print(f"\n[2A] Dropped rows with no PM2.5: {before - len(df):,}")
print(f"[2A] Rows left: {len(df):,}")

print("\nSTEP 2A DONE")
print("=" * 70)

STEP 2A: CLEAN INVALID VALUES
     pm25  : removed 7 bad values
     pm10  : removed 3 bad values
     no2   : removed 0 bad values
     co    : removed 0 bad values
     o3    : removed 0 bad values

[2A] Dropped rows with no PM2.5: 806,997
[2A] Rows left: 6,013,836

STEP 2A DONE


In [ ]:
print("=" * 70)
print("STEP 2B: STATION COVERAGE")
print("=" * 70)

MIN_COVERAGE = 0.50

date_min = df["timestamp"].min()
date_max = df["timestamp"].max()

total_hours = len(
    pd.date_range(date_min, date_max, freq="h")
)

readings = df.groupby("station")["pm25"].count()

coverage = readings / total_hours

print(f"\n[2B] Date range: {date_min} -> {date_max}")
print(f"[2B] Hours in the date range: {total_hours:,}")

print("\n[2B] Coverage per station:")

for name, pct in coverage.sort_values(
    ascending=False
).items():

    flag = "keep" if pct >= MIN_COVERAGE else "DROP"

    print(
        f"     {name[:46]:46} "
        f"{pct * 100:5.1f}%   {flag}"
    )

good_stations = sorted(
    coverage[coverage >= MIN_COVERAGE].index
)

df = df[
    df["station"].isin(good_stations)
].copy()

print(
    f"\n[2B] Keeping "
    f"{len(good_stations)} of "
    f"{len(coverage)} stations."
)

print(f"[2B] Rows now: {len(df):,}")

if len(good_stations) < 5:
    raise RuntimeError(
        "Fewer than 5 stations survived. "
        "Check Step 2A."
    )

print("\nSTEP 2B DONE")
print("=" * 70)

STEP 2B: STATION COVERAGE

[2B] Date range: 2019-01-01 05:30:00 -> 2024-01-01 05:15:00
[2B] Hours in the date range: 43,824

[2B] Coverage per station:
     Okhla Phase-2, Delhi - DPCC                    388.8%   keep
     Major Dhyan Chand National Stadium, Delhi - DP 388.1%   keep
     Sri Aurobindo Marg, Delhi - DPCC               387.8%   keep
     Dwarka-Sector 8, Delhi - DPCC                  386.6%   keep
     Nehru Nagar, Delhi - DPCC                      385.4%   keep
     Bawana, Delhi - DPCC                           382.9%   keep
     CRRI Mathura Road, Delhi - IMD                 382.3%   keep
     Patparganj, Delhi - DPCC                       381.6%   keep
     Sonia Vihar, Delhi - DPCC                      381.5%   keep
     Mundka, Delhi - DPCC                           380.0%   keep
     Jawaharlal Nehru Stadium, Delhi - DPCC         379.6%   keep
     Dr. Karni Singh Shooting Range, Delhi - DPCC   377.0%   keep
     Wazirpur, Delhi - DPCC                         376.

In [ ]:
print("=" * 70)
print("STEP 2C: COMPLETE HOURLY TIMELINE")
print("=" * 70)

FFILL_LIMIT = 3

all_hours = pd.date_range(
    df["timestamp"].min(),
    df["timestamp"].max(),
    freq="h"
)

full_index = pd.MultiIndex.from_product(
    [good_stations, all_hours],
    names=["station", "timestamp"]
)

print(
    f"\n[2C] {len(good_stations)} stations x "
    f"{len(all_hours):,} hours "
    f"= {len(full_index):,} slots"
)

df = (
    df.set_index(["station", "timestamp"])
      .reindex(full_index)
      .reset_index()
)

print("\n[2C] Forward-filling gaps of up to 3 hours...")

for col in POLLUTANT_COLS:
    before = int(df[col].isna().sum())

    df[col] = (
        df.groupby("station")[col]
          .ffill(limit=FFILL_LIMIT)
    )

    after = int(df[col].isna().sum())

    print(
        f"     {col:6s}: filled "
        f"{before - after:,} | "
        f"still blank {after:,}"
    )

print(
    f"\n[2C] Records now: {len(df):,} rows "
    f"({df['station'].nunique()} stations x "
    f"{df['timestamp'].nunique():,} hours)"
)

gc.collect()

print("\nSTEP 2C DONE")
print("=" * 70)

STEP 2C: COMPLETE HOURLY TIMELINE

[2C] 39 stations x 43,824 hours = 1,709,136 slots

[2C] Forward-filling gaps of up to 3 hours...
     pm25  : filled 53,420 | still blank 143,175
     pm10  : filled 61,106 | still blank 209,655
     no2   : filled 102,937 | still blank 180,351
     co    : filled 148,791 | still blank 183,246
     o3    : filled 120,282 | still blank 189,560

[2C] Records now: 1,709,136 rows (39 stations x 43,824 hours)

STEP 2C DONE


In [ ]:
print("=" * 70)
print("CHECK BEFORE STEP 6B")
print("=" * 70)

required = [
    "X_train",
    "y_train",
    "X_val",
    "y_val",
    "X_test",
    "y_test",
    "LOOKBACK",
    "HORIZONS",
    "RANDOM_SEED",
]

for name in required:
    print(f"{name:15s} :", "READY" if name in globals() else "MISSING")

CHECK BEFORE STEP 6B
X_train         : READY
y_train         : READY
X_val           : READY
y_val           : READY
X_test          : READY
y_test          : READY
LOOKBACK        : MISSING
HORIZONS        : MISSING
RANDOM_SEED     : MISSING


In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [ ]:
LOOKBACK = 24
HORIZONS = [1, 6, 12, 24]
MAX_HORIZON = max(HORIZONS)
STRIDE = 6
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("LOOKBACK    :", LOOKBACK)
print("HORIZONS    :", HORIZONS)
print("MAX_HORIZON :", MAX_HORIZON)
print("STRIDE      :", STRIDE)
print("SEED        :", RANDOM_SEED)

LOOKBACK    : 24
HORIZONS    : [1, 6, 12, 24]
MAX_HORIZON : 24
STRIDE      : 6
SEED        : 42


In [ ]:
print("FEATURES_SCALED :", "FEATURES_SCALED" in globals())
print("FEATURE_COLS    :", "FEATURE_COLS" in globals())
print("build function  :", "build_windows_for_station" in globals())
print("assign_split    :", "assign_split" in globals())
print("good_stations   :", "good_stations" in globals())
print("df AQI          :", "aqi" in df.columns)

FEATURES_SCALED : False
FEATURE_COLS    : False
build function  : False
assign_split    : False
good_stations   : True
df AQI          : False


In [ ]:
# ============================================================
# RECOVERY — STEP 2D -> STEP 4F
# Current df is already available after Step 2C
# ============================================================

print("=" * 70)
print("RECOVERY: BUILD AQI -> FEATURES -> WINDOWS")
print("=" * 70)

import gc
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# ------------------------------------------------------------
# 1. BASIC SETTINGS
# ------------------------------------------------------------

POLLUTANT_COLS = ["pm25", "pm10", "no2", "co", "o3"]
TARGET_COL = "aqi"

LOOKBACK = 24
HORIZONS = [1, 6, 12, 24]
MAX_HORIZON = max(HORIZONS)
STRIDE = 6
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

print("\n[RECOVERY] Basic settings ready.")


# ============================================================
# STEP 2D — COMPUTE CPCB AQI
# ============================================================

def sub_index_array(values, breakpoints):

    values = np.asarray(
        values,
        dtype=np.float64
    )

    out = np.full(
        values.shape,
        np.nan,
        dtype=np.float64
    )

    valid = (
        np.isfinite(values)
        & (values >= 0)
    )

    for c_lo, c_hi, a_lo, a_hi in breakpoints:

        mask = (
            valid
            & np.isnan(out)
            & (values >= c_lo)
            & (values <= c_hi)
        )

        out[mask] = (
            a_lo
            + (a_hi - a_lo)
            * (values[mask] - c_lo)
            / (c_hi - c_lo)
        )

    top = breakpoints[-1][1]

    above = (
        valid
        & (values > top)
    )

    out[above] = 500.0

    return out


BP_PM25 = [
    (0, 30, 0, 50),
    (30, 60, 51, 100),
    (60, 90, 101, 200),
    (90, 120, 201, 300),
    (120, 250, 301, 400),
    (250, 380, 401, 500),
]

BP_PM10 = [
    (0, 50, 0, 50),
    (50, 100, 51, 100),
    (100, 250, 101, 200),
    (250, 350, 201, 300),
    (350, 430, 301, 400),
    (430, 500, 401, 500),
]

BP_NO2 = [
    (0, 40, 0, 50),
    (40, 80, 51, 100),
    (80, 180, 101, 200),
    (180, 280, 201, 300),
    (280, 400, 301, 400),
    (400, 1000, 401, 500),
]

BP_CO = [
    (0, 1.0, 0, 50),
    (1.0, 2.0, 51, 100),
    (2.0, 10.0, 101, 200),
    (10.0, 17.0, 201, 300),
    (17.0, 34.0, 301, 400),
    (34.0, 50.0, 401, 500),
]

BP_O3 = [
    (0, 50, 0, 50),
    (50, 100, 51, 100),
    (100, 168, 101, 200),
    (168, 208, 201, 300),
    (208, 748, 301, 400),
    (748, 1000, 401, 500),
]

print("\n[2D] Computing AQI...")

si_pm25 = sub_index_array(
    df["pm25"], BP_PM25
)

si_pm10 = sub_index_array(
    df["pm10"], BP_PM10
)

si_no2 = sub_index_array(
    df["no2"], BP_NO2
)

si_co = sub_index_array(
    df["co"], BP_CO
)

si_o3 = sub_index_array(
    df["o3"], BP_O3
)

df["aqi"] = np.nanmax(
    np.column_stack(
        [
            si_pm25,
            si_pm10,
            si_no2,
            si_co,
            si_o3
        ]
    ),
    axis=1
)

df.loc[
    np.isnan(si_pm25),
    "aqi"
] = np.nan

df["aqi"] = df["aqi"].clip(
    upper=500
)

print(
    f"[2D] AQI rows: "
    f"{df['aqi'].notna().sum():,}"
)

print(
    f"[2D] AQI mean: "
    f"{df['aqi'].mean():.1f}"
)


# ============================================================
# STEP 3A — TIME FEATURES
# ============================================================

def as_circle(values, period):

    angle = (
        2 * np.pi * values / period
    )

    return (
        np.sin(angle),
        np.cos(angle)
    )


ts = df["timestamp"]

df["hour_sin"], df["hour_cos"] = as_circle(
    ts.dt.hour,
    24
)

df["dow_sin"], df["dow_cos"] = as_circle(
    ts.dt.dayofweek,
    7
)

df["month_sin"], df["month_cos"] = as_circle(
    ts.dt.month - 1,
    12
)

TIME_FEATURES = [
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "month_sin",
    "month_cos"
]

print("\n[3A] Time features created.")


# ============================================================
# STEP 3B — STATION COLUMNS
# ============================================================

existing_station_cols = [
    c for c in df.columns
    if c.startswith("station_")
]

if existing_station_cols:
    df = df.drop(
        columns=existing_station_cols
    )

station_dummies = pd.get_dummies(
    df["station"],
    prefix="station",
    dtype=np.float32
)

STATION_COLS = sorted(
    station_dummies.columns
)

df = pd.concat(
    [
        df,
        station_dummies[STATION_COLS]
    ],
    axis=1
)

del station_dummies
gc.collect()

print(
    f"[3B] Station columns: "
    f"{len(STATION_COLS)}"
)


# ============================================================
# STEP 3C — FEATURE LIST
# ============================================================

FEATURE_COLS = (
    POLLUTANT_COLS
    + [TARGET_COL]
    + TIME_FEATURES
    + STATION_COLS
)

print(
    f"\n[3C] Total features: "
    f"{len(FEATURE_COLS)}"
)


# ============================================================
# STEP 4B — TIME SPLITS
# ============================================================

TRAIN_END = pd.Timestamp(
    "2022-12-31 23:59:59"
)

VAL_START = pd.Timestamp(
    "2023-01-01"
)

VAL_END = pd.Timestamp(
    "2023-06-30 23:59:59"
)

TEST_START = pd.Timestamp(
    "2023-07-01"
)


def assign_split(when):

    if when <= TRAIN_END:
        return "train"

    if (
        VAL_START
        <= when
        <= VAL_END
    ):
        return "val"

    return "test"


print("\n[4B] Split function ready.")


# ============================================================
# STEP 4C — SCALE FEATURES
# ============================================================

train_rows = (
    df["timestamp"]
    <= TRAIN_END
)

complete = (
    df[FEATURE_COLS]
    .notna()
    .all(axis=1)
)

fit_mask = (
    train_rows
    & complete
)

scaler = MinMaxScaler()

scaler.fit(
    df.loc[
        fit_mask,
        FEATURE_COLS
    ]
)

FEATURES_SCALED = np.full(
    (
        len(df),
        len(FEATURE_COLS)
    ),
    np.nan,
    dtype=np.float32
)

FEATURES_SCALED[
    complete.to_numpy()
] = scaler.transform(
    df.loc[
        complete,
        FEATURE_COLS
    ]
).astype(np.float32)

print(
    f"\n[4C] Scaled rows: "
    f"{int(complete.sum()):,}"
)


# ============================================================
# STEP 4D — WINDOW BUILDER
# ============================================================

station_of_row = (
    df["station"].to_numpy()
)

aqi_of_row = (
    df["aqi"].to_numpy(
        dtype=np.float32
    )
)

time_of_row = (
    df["timestamp"].to_numpy()
)


def build_windows_for_station(name):

    rows = np.where(
        station_of_row == name
    )[0]

    aqi = aqi_of_row[rows]
    times = time_of_row[rows]

    X = []
    y = []
    end_times = []
    last_aqi = []

    last_possible_start = (
        len(rows)
        - LOOKBACK
        - MAX_HORIZON
    )

    for start in range(
        0,
        last_possible_start + 1,
        STRIDE
    ):

        end = (
            start
            + LOOKBACK
            - 1
        )

        history = FEATURES_SCALED[
            rows[start:end + 1]
        ]

        if np.isnan(history).any():
            continue

        targets = np.array(
            [
                aqi[end + h]
                for h in HORIZONS
            ],
            dtype=np.float32
        )

        if np.isnan(targets).any():
            continue

        X.append(history)
        y.append(targets)
        end_times.append(times[end])
        last_aqi.append(aqi[end])

    if not X:
        return None

    return (
        np.stack(X).astype(
            np.float32
        ),
        np.stack(y).astype(
            np.float32
        ),
        np.array(end_times),
        np.array(
            last_aqi,
            dtype=np.float32
        )
    )


print(
    "\n[4D] Window builder ready."
)


# ============================================================
# STEP 4E — BUILD WINDOWS
# ============================================================

buckets = {
    split: {
        "X": [],
        "y": [],
        "end": [],
        "last": [],
        "station": []
    }
    for split in (
        "train",
        "val",
        "test"
    )
}

print("\n[4E] Building windows...")

for name in good_stations:

    result = build_windows_for_station(
        name
    )

    if result is None:
        continue

    X_station, y_station, ends, lasts = result

    labels = np.array([
        assign_split(
            pd.Timestamp(t)
        )
        for t in ends
    ])

    for split in (
        "train",
        "val",
        "test"
    ):

        pick = (
            labels == split
        )

        if not pick.any():
            continue

        buckets[split]["X"].append(
            X_station[pick]
        )

        buckets[split]["y"].append(
            y_station[pick]
        )

        buckets[split]["end"].append(
            ends[pick]
        )

        buckets[split]["last"].append(
            lasts[pick]
        )

        buckets[split]["station"].append(
            np.full(
                pick.sum(),
                name,
                dtype=object
            )
        )

    del X_station
    del y_station
    del ends
    del lasts
    del result

gc.collect()

print("[4E] Windows created.")


# ============================================================
# STEP 4F — ASSEMBLE
# ============================================================

def join(split, key):

    pieces = buckets[split].pop(
        key,
        None
    )

    if not pieces:
        return None

    return np.concatenate(
        pieces,
        axis=0
    )


X_train = join(
    "train", "X"
)

y_train = join(
    "train", "y"
)

X_val = join(
    "val", "X"
)

y_val = join(
    "val", "y"
)

X_test = join(
    "test", "X"
)

y_test = join(
    "test", "y"
)

end_train = join(
    "train", "end"
)

end_val = join(
    "val", "end"
)

end_test = join(
    "test", "end"
)

station_test = join(
    "test", "station"
)

last_aqi_val = join(
    "val", "last"
)

last_aqi_test = join(
    "test", "last"
)

print("\n[4F] Checking splits...")

for name, array in [
    ("Train", X_train),
    ("Val", X_val),
    ("Test", X_test)
]:

    if array is None:
        raise RuntimeError(
            f"{name} windows are empty."
        )

    print(
        f"     {name:5s}: "
        f"{array.shape}"
    )

print("\n")
print("RECOVERY COMPLETE")
print("=" * 70)

RECOVERY: BUILD AQI -> FEATURES -> WINDOWS

[RECOVERY] Basic settings ready.

[2D] Computing AQI...


/tmp/ipykernel_1021/957250281.py:151: RuntimeWarning: All-NaN slice encountered
  df["aqi"] = np.nanmax(


[2D] AQI rows: 1,565,961
[2D] AQI mean: 215.3

[3A] Time features created.
[3B] Station columns: 39

[3C] Total features: 51

[4B] Split function ready.

[4C] Scaled rows: 1,406,761

[4D] Window builder ready.

[4E] Building windows...
[4E] Windows created.

[4F] Checking splits...
     Train: (144825, 24, 51)
     Val  : (19443, 24, 51)
     Test : (18355, 24, 51)


RECOVERY COMPLETE


In [ ]:
print("=" * 70)
print("STEP 6B: BUILD THE MODEL")
print("=" * 70)

n_features = X_train.shape[-1]

print(
    f"\n[6B] Input per sample: "
    f"{LOOKBACK} hours x {n_features} features"
)

keras.backend.clear_session()
tf.random.set_seed(RANDOM_SEED)

model = keras.Sequential(
    [
        layers.Input(
            shape=(LOOKBACK, n_features),
            name="past_24h"
        ),
        layers.LSTM(
            96,
            return_sequences=True,
            name="lstm_1"
        ),
        layers.Dropout(
            0.2,
            name="dropout_1"
        ),
        layers.LSTM(
            48,
            name="lstm_2"
        ),
        layers.Dropout(
            0.2,
            name="dropout_2"
        ),
        layers.Dense(
            32,
            activation="relu",
            name="dense_hidden"
        ),
        layers.Dense(
            len(HORIZONS),
            name="aqi_horizons"
        ),
    ],
    name="vayupath_aqi_lstm",
)

model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="mse",
    metrics=["mae"],
)

print("\n[6B] Layers:")
model.summary()

print("\nSTEP 6B DONE")
print("=" * 70)

STEP 6B: BUILD THE MODEL

[6B] Input per sample: 24 hours x 51 features

[6B] Layers:


Model: "vayupath_aqi_lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 24, 96)         │        56,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 24, 96)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 48)             │        27,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 48)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden (Dense)            │ (None, 32)             │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ aqi_horizons (Dense)            │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 86,372 (337.39 KB)

 Trainable params: 86,372 (337.39 KB)

 Non-trainable params: 0 (0.00 B)


STEP 6B DONE


In [7]:
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from pathlib import Path


In [17]:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

NameError: name 'X_train' is not defined